<a href="https://colab.research.google.com/github/projectapertureBSM/Beyond-Standard-Model-/blob/main/Latex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
!apt-get update
!apt-get install -y texlive-full ghostscript poppler-utils
!pip install tqdm colorama

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,720 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,245 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Pac

In [ ]:
# Advanced AI Toolkit: Mistral 7B + Neural Networks & Signal Processing
# This notebook provides a comprehensive AI toolkit with language models, neural networks, and signal processing

# 0. Install all required libraries
!pip install -q transformers accelerate bitsandbytes sentencepiece torch torchvision torchaudio matplotlib numpy scipy scikit-learn opencv-python filterpy

# 1. Import common libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import gc
from IPython.display import clear_output, display, HTML
import ipywidgets as widgets
from sklearn.decomposition import PCA, FastICA
from scipy import signal
import cv2
from filterpy.kalman import KalmanFilter
import io
import base64
from google.colab import drive
import os
import copy

# 2. Mount Google Drive (for saving models and data)
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/AI_Toolkit', exist_ok=True)

# 3. Check GPU availability
print("Checking GPU availability...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi

# ====================================================================
# PART 1: LANGUAGE MODEL (MISTRAL 7B)
# ====================================================================

print("Setting up Mistral 7B language model...")

from transformers import AutoModelForCausalLM, AutoTokenizer

# Load Mistral 7B model (this will take a few minutes)
def load_language_model():
    print("Loading Mistral 7B (this might take a few minutes)...")
    model_name = "mistralai/Mistral-7B-Instruct-v0.2"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        load_in_4bit=True
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"Mistral 7B loaded successfully")
    return model, tokenizer

# Text generation function
def generate_text(model, tokenizer, prompt,
                 max_new_tokens=512,
                 temperature=0.7,
                 top_p=0.9,
                 top_k=40,
                 repetition_penalty=1.1,
                 do_sample=True):
    """Generate text using the Mistral 7B model"""
    # Format prompt for Mistral's expected chat format
    formatted_prompt = f"""<s>[INST] {prompt} [/INST]"""

    # Tokenize the prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    # Generate
    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )

    # Calculate generation time
    gen_time = time.time() - start_time

    # Decode the output
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract response part
    try:
        response = full_output.split("[/INST]", 1)[1].strip()
    except:
        response = full_output

    print(f"Generation took {gen_time:.2f} seconds")
    return response

# ====================================================================
# PART 2: RECURRENT NEURAL NETWORKS (RNNs)
# ====================================================================

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(GRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

# Example: Train RNN on time series data
def train_rnn(model, X_train, y_train, epochs=10, lr=0.001, batch_size=32):
    """Train RNN model on time series data"""
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).to(device)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        for batch_X, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')

    plt.figure(figsize=(10, 5))
    plt.plot(losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.show()

    return model

# Function to generate synthetic time series data for RNN training
def generate_sine_wave_data(samples=1000, seq_length=50, prediction_length=1):
    """Generate synthetic sine wave data for RNN training"""
    # Generate sine wave
    x = np.linspace(0, 50, samples + seq_length + prediction_length)
    y = np.sin(x)

    # Create sequences
    X, Y = [], []
    for i in range(samples):
        X.append(y[i:i+seq_length])
        Y.append(y[i+seq_length:i+seq_length+prediction_length])

    X = np.array(X).reshape(-1, seq_length, 1)  # Reshape for RNN input
    Y = np.array(Y).reshape(-1, prediction_length)

    # Split into train/test
    train_size = int(0.8 * len(X))
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = Y[:train_size], Y[train_size:]

    return X_train, y_train, X_test, y_test

# ====================================================================
# PART 3: CONVOLUTIONAL NEURAL NETWORKS (CNNs)
# ====================================================================

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 1000)
        self.fc2 = nn.Linear(1000, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Function to load and preprocess MNIST dataset
def load_mnist_data(batch_size=64):
    """Load MNIST dataset for CNN training"""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = torchvision.datasets.MNIST(root='./data',
                                              train=True,
                                              transform=transform,
                                              download=True)

    test_dataset = torchvision.datasets.MNIST(root='./data',
                                             train=False,
                                             transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

# Function to train CNN model
def train_cnn(model, train_loader, epochs=5, lr=0.001):
    """Train CNN model on image data"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            if batch_idx % 100 == 99:    # print every 100 mini-batches
                print(f'[Epoch {epoch + 1}, Batch {batch_idx + 1}] Loss: {running_loss / 100:.3f} | Acc: {100. * correct / total:.3f}%')
                running_loss = 0.0

    print('Finished Training')
    return model

# ====================================================================
# PART 4: GENERATIVE ADVERSARIAL NETWORKS (GANs)
# ====================================================================

class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_shape=(1, 28, 28)):
        super(Generator, self).__init__()
        self.img_shape = img_shape

        def block(in_feat, out_feat, normalize=True):
            layers = [nn.Linear(in_feat, out_feat)]
            if normalize:
                layers.append(nn.BatchNorm1d(out_feat, 0.8))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(latent_dim, 128, normalize=False),
            *block(128, 256),
            *block(256, 512),
            *block(512, 1024),
            nn.Linear(1024, int(np.prod(img_shape))),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), *self.img_shape)
        return img

class Discriminator(nn.Module):
    def __init__(self, img_shape=(1, 28, 28)):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(int(np.prod(img_shape)), 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

# Function to train a GAN on MNIST
def train_gan(generator, discriminator, dataloader, epochs=10, latent_dim=100, lr=0.0002, b1=0.5, b2=0.999, sample_interval=400):
    """Train a GAN on image data"""
    generator = generator.to(device)
    discriminator = discriminator.to(device)

    adversarial_loss = torch.nn.BCELoss()

    optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr, betas=(b1, b2))
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(b1, b2))

    for epoch in range(epochs):
        for i, (imgs, _) in enumerate(dataloader):

            # Configure real and fake labels
            valid = torch.ones(imgs.size(0), 1).to(device)
            fake = torch.zeros(imgs.size(0), 1).to(device)

            # Configure input
            real_imgs = imgs.to(device)

            # -----------------
            #  Train Generator
            # -----------------
            optimizer_G.zero_grad()

            # Sample noise as generator input
            z = torch.randn(imgs.size(0), latent_dim).to(device)

            # Generate a batch of images
            gen_imgs = generator(z)

            # Loss measures generator's ability to fool the discriminator
            g_loss = adversarial_loss(discriminator(gen_imgs), valid)

            g_loss.backward()
            optimizer_G.step()

            # ---------------------
            #  Train Discriminator
            # ---------------------
            optimizer_D.zero_grad()

            # Measure discriminator's ability to classify real from generated samples
            real_loss = adversarial_loss(discriminator(real_imgs), valid)
            fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            optimizer_D.step()

            if i % 200 == 0:
                print(f"[Epoch {epoch}/{epochs}] [Batch {i}/{len(dataloader)}] [D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")

            batches_done = epoch * len(dataloader) + i
            if batches_done % sample_interval == 0:
                # Display generated images
                save_image_grid(gen_imgs.data[:25], f"GAN_images/epoch_{epoch}_batch_{batches_done}.png", nrow=5, normalize=True)

    return generator, discriminator

# Function to save image grid
def save_image_grid(images, path, nrow=10, normalize=True):
    """Save a grid of images to a file"""
    # Create a directory if it doesn't exist
    os.makedirs(os.path.dirname(path), exist_ok=True)

    # Convert to PIL images
    grid = torchvision.utils.make_grid(images, nrow=nrow, normalize=normalize)
    np_grid = grid.cpu().detach().numpy().transpose((1, 2, 0))

    if normalize:
        np_grid = (np_grid * 255).astype(np.uint8)

    # Save image
    plt.figure(figsize=(10, 10))
    plt.imshow(np_grid)
    plt.axis('off')
    plt.savefig(path)
    plt.close()

# Function to visualize generated images
def visualize_gan_output(generator, latent_dim=100, n_samples=25):
    """Visualize output from a GAN generator"""
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, latent_dim).to(device)
        gen_imgs = generator(z)

        # Display images
        grid = torchvision.utils.make_grid(gen_imgs, nrow=5, normalize=True)
        np_grid = grid.cpu().detach().numpy().transpose((1, 2, 0))

        plt.figure(figsize=(10, 10))
        plt.imshow(np_grid)
        plt.axis('off')
        plt.show()

# ====================================================================
# PART 5: INDEPENDENT COMPONENT ANALYSIS (ICA)
# ====================================================================

def perform_ica(X, n_components=None):
    """Perform ICA on data X"""
    ica = FastICA(n_components=n_components, random_state=42)
    S = ica.fit_transform(X)
    A = ica.mixing_
    return S, A, ica

# Generate synthetic data for ICA
def generate_ica_example():
    """Generate synthetic data for ICA demonstration"""
    np.random.seed(42)
    n_samples = 2000

    # Generate sources
    time = np.linspace(0, 8, n_samples)
    s1 = np.sin(2 * time)  # Sinusoidal source
    s2 = np.sign(np.sin(3 * time))  # Square wave source
    s3 = np.random.laplace(0, 1, size=n_samples)  # Laplace source

    # Create source matrix
    S = np.c_[s1, s2, s3]

    # Mixing matrix
    A = np.array([[1, 0.5, 0.3], [0.5, 1, 0.2], [0.2, 0.3, 1]])

    # Mix the sources
    X = np.dot(S, A.T)

    # Visualize the mixed signals
    plt.figure(figsize=(10, 6))
    plt.subplot(3, 1, 1)
    plt.plot(X[:, 0])
    plt.title('Mixed Signal 1')

    plt.subplot(3, 1, 2)
    plt.plot(X[:, 1])
    plt.title('Mixed Signal 2')

    plt.subplot(3, 1, 3)
    plt.plot(X[:, 2])
    plt.title('Mixed Signal 3')
    plt.tight_layout()
    plt.show()

    # Perform ICA
    S_ica, A_ica, ica_model = perform_ica(X)

    # Visualize the recovered signals
    plt.figure(figsize=(10, 6))
    plt.subplot(3, 1, 1)
    plt.plot(S_ica[:, 0])
    plt.title('Recovered Signal 1')

    plt.subplot(3, 1, 2)
    plt.plot(S_ica[:, 1])
    plt.title('Recovered Signal 2')

    plt.subplot(3, 1, 3)
    plt.plot(S_ica[:, 2])
    plt.title('Recovered Signal 3')
    plt.tight_layout()
    plt.show()

    return X, S_ica, ica_model

# ====================================================================
# PART 6: PRINCIPAL COMPONENT ANALYSIS (PCA)
# ====================================================================

def perform_pca(X, n_components=None):
    """Perform PCA on data X"""
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)
    return X_pca, pca

# Function to visualize PCA results
def visualize_pca(X, y, n_components=2):
    """Visualize PCA results"""
    X_pca, pca = perform_pca(X, n_components)

    plt.figure(figsize=(10, 8))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=50, alpha=0.8)
    plt.colorbar()
    plt.title(f'PCA Visualization (explained variance: {pca.explained_variance_ratio_.sum():.2f})')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2f})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2f})')
    plt.show()

    # Plot explained variance
    plt.figure(figsize=(10, 6))
    plt.plot(np.cumsum(pca.explained_variance_ratio_))
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.title('Explained Variance vs. Number of Components')
    plt.grid(True)
    plt.show()

    return X_pca, pca

# ====================================================================
# PART 7: MULTI-TARGET TRACKING
# ====================================================================

class KalmanTracker:
    """Kalman Filter-based tracker for multi-target tracking"""
    def __init__(self, initial_state, dt=1.0):
        # Initialize the Kalman Filter with state vector [x, y, vx, vy]
        self.kf = KalmanFilter(dim_x=4, dim_z=2)

        # State transition matrix
        self.kf.F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ])

        # Measurement function (we only measure position [x, y])
        self.kf.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])

        # Covariance matrix
        self.kf.P *= 10

        # Measurement noise
        self.kf.R = np.array([
            [1, 0],
            [0, 1]
        ])

        # Process noise
        self.kf.Q = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ]) * 0.01

        # Initialize state
        self.kf.x = np.array(initial_state).reshape(4, 1)

        # Track history
        self.history = [initial_state[:2]]

        # Track ID
        self.id = np.random.randint(1000)

    def predict(self):
        """Predict next state"""
        self.kf.predict()
        return self.kf.x[:2].flatten()

    def update(self, measurement):
        """Update state with measurement"""
        self.kf.update(measurement)
        self.history.append(self.kf.x[:2].flatten())
        return self.kf.x[:2].flatten()

# Function to generate synthetic tracking data
def generate_tracking_data(num_tracks=3, num_frames=100, noise_level=5):
    """Generate synthetic tracking data with multiple targets"""
    np.random.seed(42)
    tracks = []

    for i in range(num_tracks):
        # Create a track with random starting point and velocity
        x0 = np.random.randint(50, 450)
        y0 = np.random.randint(50, 450)

        vx = np.random.uniform(-3, 3)
        vy = np.random.uniform(-3, 3)

        track = []
        x, y = x0, y0

        for t in range(num_frames):
            # Add some random acceleration
            vx += np.random.uniform(-0.1, 0.1)
            vy += np.random.uniform(-0.1, 0.1)

            # Update position
            x += vx
            y += vy

            # Bounce off "walls"
            if x < 0 or x > 500:
                vx = -vx
            if y < 0 or y > 500:
                vy = -vy

            # Add measurement noise
            measurement = [x + np.random.normal(0, noise_level),
                          y + np.random.normal(0, noise_level)]
            track.append(measurement)

        tracks.append(track)

    return tracks

# Function to demonstrate multi-target tracking
def demonstrate_tracking():
    """Demonstrate multi-target tracking with Kalman filters"""
    # Generate synthetic tracks
    tracks = generate_tracking_data(num_tracks=3, num_frames=50, noise_level=5)

    # Initialize trackers
    trackers = []
    for track in tracks:
        initial_state = [track[0][0], track[0][1], 0, 0]  # [x, y, vx, vy]
        trackers.append(KalmanTracker(initial_state))

    # Colors for tracks
    colors = ['r', 'g', 'b']

    # Track and visualize
    plt.figure(figsize=(10, 8))

    # Plot ground truth
    for i, track in enumerate(tracks):
        track_array = np.array(track)
        plt.plot(track_array[:, 0], track_array[:, 1], 'o', color=colors[i], alpha=0.3)

    # Perform tracking
    for frame in range(1, len(tracks[0])):
        for i, tracker in enumerate(trackers):
            # Predict
            prediction = tracker.predict()

            # Update with measurement
            measurement = tracks[i][frame]
            estimated_pos = tracker.update(measurement)

    # Plot estimated tracks
    for i, tracker in enumerate(trackers):
        track_history = np.array(tracker.history)
        plt.plot(track_history[:, 0], track_history[:, 1], '-', color=colors[i], linewidth=2,
                label=f'Tracker {i+1}')

    plt.grid(True)
    plt.legend()
    plt.title('Multi-Target Tracking with Kalman Filters')
    plt.xlabel('X Position')
    plt.ylabel('Y Position')
    plt.show()

    return trackers

# ====================================================================
# PART 8: POWER SPECTRAL DENSITY (PSD)
# ====================================================================

def compute_psd(signal_data, fs=1000.0):
    """Compute Power Spectral Density of a signal"""
    # Compute PSD using Welch's method
    frequencies, psd = signal.welch(signal_data, fs=fs, nperseg=1024)

    plt.figure(figsize=(10, 6))
    plt.semilogy(frequencies, psd)
    plt.title('Power Spectral Density')
    plt.xlabel('Frequency [Hz]')
    plt.ylabel('PSD [V^2/Hz]')
    plt.grid(True)
    plt.show()

    return frequencies, psd

# Generate example signal for PSD analysis
def generate_psd_example():
    """Generate example signal with multiple frequency components"""
    # Parameters
    fs = 1000.0  # Sampling frequency
    duration = 5.0  # Duration in seconds

    # Generate time array
    t = np.arange(0, duration, 1/fs)

    # Generate signal with multiple frequency components
    f1, f2, f3 = 10, 50, 120  # Frequencies in Hz
    signal_data = (
        np.sin(2 * np.pi * f1 * t) +
        0.5 * np.sin(2 * np.pi * f2 * t) +
        0.3 * np.sin(2 * np.pi * f3 * t) +
        0.1 * np.random.randn(len(t))  # Add some noise
    )

    # Plot the signal
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot

In [ ]:
# Advanced Particle Physics Data Processor
# For use with Minstrel 7 in Google Colab

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
from collections import defaultdict
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, KMeans

# For PDF processing
!pip install -q PyPDF2 pdfplumber tabula-py pdf2image pytesseract
import PyPDF2
import pdfplumber
import tabula
from pdf2image import convert_from_path
import pytesseract

# For physics-specific libraries
!pip install -q particle iminuit hepunits
import particle
from particle import Particle, PDGID

# Set up visualization style
plt.style.use('ggplot')
sns.set_context("notebook", font_scale=1.5)
warnings.filterwarnings('ignore')

# Utility function to ensure directories exist
def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)
    return directory

# Set up working directories
BASE_DIR = "/content"
DATA_DIR = ensure_dir(os.path.join(BASE_DIR, "data"))
PDF_DIR = ensure_dir(os.path.join(DATA_DIR, "pdfs"))
CSV_DIR = ensure_dir(os.path.join(DATA_DIR, "csvs"))
OUTPUT_DIR = ensure_dir(os.path.join(BASE_DIR, "output"))
MODEL_DIR = ensure_dir(os.path.join(BASE_DIR, "models"))

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_PATH = "/content/drive/MyDrive/Minstrel7_ParticlePhysics"
ensure_dir(GDRIVE_PATH)

print("✅ Environment setup complete")
print("📁 Working directories created")
print("💾 Google Drive mounted")

# ====== PARTICLE ALIAS MANAGEMENT SYSTEM ======

class ParticleAliasManager:
    """Manages particle name aliases and standardization for physics data processing"""

    def __init__(self):
        self.alias_graph = nx.Graph()
        self.canonical_names = {}
        self.pdg_id_map = {}
        self._initialize_standard_particles()
        self._load_user_aliases()

    def _initialize_standard_particles(self):
        """Initialize with standard particle data from the particle package"""
        print("📊 Initializing standard particle database...")

        # Map PDG IDs to canonical names and aliases
        for p in tqdm(Particle.findall(lambda p: p.pdgid.is_valid), desc="Loading particles"):
            if not p.name:
                continue

            canonical = p.name
            self.canonical_names[canonical] = canonical
            self.pdg_id_map[canonical] = int(p.pdgid)

            # Add node for canonical name
            self.alias_graph.add_node(canonical, is_canonical=True, pdg_id=int(p.pdgid))

            # Add all known aliases
            aliases = []
            if p.latex_name and p.latex_name != p.name:
                aliases.append(p.latex_name)
            if p.html_name and p.html_name != p.name:
                aliases.append(p.html_name)

            # Add common variations
            if "(" in canonical:
                aliases.append(canonical.replace("(", "").replace(")", ""))

            # Handle antiparticles consistently
            if p.pdgid.has_antiparticle:
                if "bar" in canonical:
                    aliases.append(canonical.replace("bar", "~"))
                    aliases.append("anti-" + canonical.replace("bar", ""))
                if p.pdgid < 0:
                    base_name = Particle.from_pdgid(-p.pdgid).name
                    if base_name:
                        aliases.append(f"anti-{base_name}")
                        aliases.append(f"~{base_name}")

            # Add all aliases as edges to the canonical name
            for alias in aliases:
                if alias and isinstance(alias, str):
                    self.alias_graph.add_node(alias, is_canonical=False)
                    self.alias_graph.add_edge(canonical, alias)
                    self.canonical_names[alias] = canonical

        print(f"✅ Loaded {len(self.canonical_names)} particle names and aliases")

    def _load_user_aliases(self):
        """Load user-defined aliases from JSON file if available"""
        alias_file = os.path.join(GDRIVE_PATH, "particle_aliases.json")
        if os.path.exists(alias_file):
            with open(alias_file, 'r') as f:
                user_aliases = json.load(f)

            for canonical, aliases in user_aliases.items():
                # Add canonical name if not already present
                if canonical not in self.canonical_names:
                    self.alias_graph.add_node(canonical, is_canonical=True)
                    self.canonical_names[canonical] = canonical

                # Add aliases
                for alias in aliases:
                    self.alias_graph.add_node(alias, is_canonical=False)
                    self.alias_graph.add_edge(canonical, alias)
                    self.canonical_names[alias] = canonical

            print(f"✅ Loaded user-defined aliases from {alias_file}")
        else:
            print("ℹ️ No user-defined aliases found. Creating a new file for future use.")
            # Create a template file
            example = {
                "pion+": ["pi+", "pi_plus", "positive_pion"],
                "pion-": ["pi-", "pi_minus", "negative_pion"]
            }
            with open(alias_file, 'w') as f:
                json.dump(example, f, indent=2)

    def add_alias(self, canonical_name, alias):
        """Add a new alias for a particle"""
        if canonical_name not in self.alias_graph:
            self.alias_graph.add_node(canonical_name, is_canonical=True)
            self.canonical_names[canonical_name] = canonical_name

        self.alias_graph.add_node(alias, is_canonical=False)
        self.alias_graph.add_edge(canonical_name, alias)
        self.canonical_names[alias] = canonical_name

        # Save the updated aliases
        self._save_user_aliases()
        return True

    def _save_user_aliases(self):
        """Save user-defined aliases to JSON file"""
        alias_file = os.path.join(GDRIVE_PATH, "particle_aliases.json")

        # Extract user-defined aliases (not in standard particle database)
        user_aliases = defaultdict(list)
        for canonical, aliases in self._get_all_aliases().items():
            if not self._is_standard_particle(canonical):
                user_aliases[canonical] = aliases

        with open(alias_file, 'w') as f:
            json.dump(user_aliases, f, indent=2)

    def _is_standard_particle(self, name):
        """Check if a particle name is from the standard particle database"""
        try:
            Particle.find(name)
            return True
        except:
            return False

    def get_canonical_name(self, particle_name):
        """Get the canonical name for a given particle name or alias"""
        if particle_name in self.canonical_names:
            return self.canonical_names[particle_name]

        # Try case-insensitive match
        lower_name = particle_name.lower()
        for name in self.canonical_names:
            if name.lower() == lower_name:
                return self.canonical_names[name]

        # Try matching with different formatting
        cleaned_name = re.sub(r'[^a-zA-Z0-9]+', '', particle_name.lower())
        for name in self.canonical_names:
            if re.sub(r'[^a-zA-Z0-9]+', '', name.lower()) == cleaned_name:
                return self.canonical_names[name]

        return None

    def get_pdg_id(self, particle_name):
        """Get the PDG ID for a given particle name or alias"""
        canonical = self.get_canonical_name(particle_name)
        if canonical and canonical in self.pdg_id_map:
            return self.pdg_id_map[canonical]
        return None

    def _get_all_aliases(self):
        """Get all aliases grouped by canonical name"""
        result = defaultdict(list)
        for node in self.alias_graph.nodes():
            if self.alias_graph.nodes[node].get('is_canonical', False):
                # Find all neighbors (aliases)
                for neighbor in self.alias_graph.neighbors(node):
                    result[node].append(neighbor)
        return result

    def visualize_aliases(self):
        """Visualize the alias graph"""
        plt.figure(figsize=(12, 10))

        # Create a subgraph with a reasonable number of nodes for visualization
        main_particles = ["electron", "muon", "tau", "proton", "neutron", "pion+", "pion-", "pion0"]
        subgraph_nodes = set()

        for p in main_particles:
            if p in self.alias_graph:
                subgraph_nodes.add(p)
                for neighbor in self.alias_graph.neighbors(p):
                    subgraph_nodes.add(neighbor)

        # Create subgraph
        subgraph = self.alias_graph.subgraph(subgraph_nodes)

        # Set up node colors
        node_colors = []
        for node in subgraph.nodes():
            if subgraph.nodes[node].get('is_canonical', False):
                node_colors.append('lightblue')
            else:
                node_colors.append('lightgreen')

        # Draw the graph
        pos = nx.spring_layout(subgraph, seed=42)
        nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors, alpha=0.8)
        nx.draw_networkx_edges(subgraph, pos, alpha=0.5)
        nx.draw_networkx_labels(subgraph, pos, font_size=10)

        plt.title("Particle Name and Alias Relationships")
        plt.axis('off')
        plt.savefig(os.path.join(OUTPUT_DIR, 'particle_aliases.png'), dpi=300, bbox_inches='tight')
        plt.close()

# ====== PDF PROCESSING SYSTEM ======

class PDFProcessor:
    """Advanced PDF processor for extracting particle physics data from PDFs"""

    def __init__(self, alias_manager):
        self.alias_manager = alias_manager

    def extract_data_from_pdf(self, pdf_path):
        """Extract data from PDF using multiple methods and combine results"""
        print(f"📄 Processing PDF: {os.path.basename(pdf_path)}")
        results = {
            'tables': [],
            'text': '',
            'particle_mentions': defaultdict(int),
            'metadata': {}
        }

        # Get basic PDF metadata
        with open(pdf_path, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            results['metadata']['pages'] = len(pdf_reader.pages)
            if pdf_reader.metadata:
                for key, value in pdf_reader.metadata.items():
                    if key.startswith('/'):
                        clean_key = key[1:]
                    else:
                        clean_key = key
                    results['metadata'][clean_key] = value

        # Method 1: Use tabula for table extraction
        try:
            print("  Extracting tables with tabula...")
            tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True)
            for i, table in enumerate(tables):
                if not table.empty:
                    results['tables'].append({
                        'method': 'tabula',
                        'page': i+1,  # Approximate page number
                        'data': table
                    })
            print(f"  ✅ Extracted {len(tables)} tables with tabula")
        except Exception as e:
            print(f"  ⚠️ Error extracting tables with tabula: {e}")

        # Method 2: Use pdfplumber for more structured extraction
        try:
            with pdfplumber.open(pdf_path) as pdf:
                all_text = []
                for i, page in enumerate(tqdm(pdf.pages, desc="  Extracting with pdfplumber")):
                    # Extract text
                    text = page.extract_text()
                    if text:
                        all_text.append(text)

                    # Extract tables
                    tables = page.extract_tables()
                    for table in tables:
                        if table and any(table):
                            # Convert to pandas DataFrame
                            df = pd.DataFrame(table[1:], columns=table[0] if table[0] else None)
                            results['tables'].append({
                                'method': 'pdfplumber',
                                'page': i+1,
                                'data': df
                            })

                # Combine all text
                results['text'] = '\n'.join(all_text)

            print(f"  ✅ Extracted text and {len(results['tables']) - len(tables)} additional tables with pdfplumber")
        except Exception as e:
            print(f"  ⚠️ Error extracting with pdfplumber: {e}")

        # Find particle mentions in the text
        if results['text']:
            # Get all particle names from the alias manager
            all_particles = list(self.alias_manager.canonical_names.keys())

            # Sort by length (descending) to match longest names first
            all_particles.sort(key=len, reverse=True)

            for particle in all_particles:
                # Count case-insensitive mentions
                count = len(re.findall(r'\b' + re.escape(particle) + r'\b', results['text'], re.IGNORECASE))
                if count > 0:
                    canonical = self.alias_manager.get_canonical_name(particle)
                    results['particle_mentions'][canonical] += count

            print(f"  ✅ Found mentions of {len(results['particle_mentions'])} different particles")

        return results

    def process_all_pdfs(self, directory=PDF_DIR):
        """Process all PDFs in the given directory"""
        pdf_files = [f for f in os.listdir(directory) if f.lower().endswith('.pdf')]

        if not pdf_files:
            print("❌ No PDF files found in the directory")
            return {}

        results = {}
        for pdf_file in pdf_files:
            pdf_path = os.path.join(directory, pdf_file)
            results[pdf_file] = self.extract_data_from_pdf(pdf_path)

        # Save the processed results
        with open(os.path.join(OUTPUT_DIR, 'pdf_extraction_results.json'), 'w') as f:
            # Convert defaultdicts to dicts for JSON serialization
            serializable_results = {}
            for pdf, data in results.items():
                serializable_results[pdf] = {
                    'tables': data['tables'],
                    'text': data['text'],
                    'particle_mentions': dict(data['particle_mentions']),
                    'metadata': data['metadata']
                }

                # Remove DataFrame objects which aren't JSON serializable
                for table in serializable_results[pdf]['tables']:
                    if 'data' in table and isinstance(table['data'], pd.DataFrame):
                        # Convert to dictionary representation
                        table['data'] = table['data'].to_dict()

            json.dump(serializable_results, f)

        print(f"✅ Processed {len(pdf_files)} PDF files and saved results")
        return results

    def visualize_particle_mentions(self, pdf_results):
        """Visualize particle mentions across all PDFs"""
        if not pdf_results:
            print("❌ No PDF results to visualize")
            return

        # Aggregate particle mentions across all PDFs
        combined_mentions = defaultdict(int)
        for pdf, data in pdf_results.items():
            for particle, count in data['particle_mentions'].items():
                combined_mentions[particle] += count

        # Get top mentioned particles
        top_particles = sorted(combined_mentions.items(), key=lambda x: x[1], reverse=True)[:20]

        # Create visualization
        plt.figure(figsize=(12, 8))
        particles, counts = zip(*top_particles) if top_particles else ([], [])

        # Create horizontal bar chart
        bars = plt.barh(particles, counts, color='skyblue')
        plt.xlabel('Number of Mentions')
        plt.title('Top Particle Mentions in PDFs')
        plt.grid(axis='x', linestyle='--', alpha=0.7)

        # Add count labels
        for bar in bars:
            width = bar.get_width()
            plt.text(width + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{width:.0f}', ha='left', va='center')

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'particle_mentions.png'), dpi=300, bbox_inches='tight')
        plt.close()

        print("✅ Created visualization of top particle mentions")

# ====== CSV ANALYSIS SYSTEM ======

class CSVAnalyzer:
    """Advanced CSV analyzer for particle physics data"""

    def __init__(self, alias_manager):
        self.alias_manager = alias_manager
        self.dataframes = {}

    def load_csv(self, csv_path, name=None):
        """Load a CSV file with intelligent handling of different formats"""
        if name is None:
            name = os.path.basename(csv_path)

        print(f"📊 Loading CSV: {name}")

        # Try different encodings and delimiters
        encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252']
        delimiters = [',', ';', '\t', '|']

        df = None
        for encoding in encodings:
            for delimiter in delimiters:
                try:
                    df = pd.read_csv(csv_path, encoding=encoding, delimiter=delimiter,
                                    on_bad_lines='skip', low_memory=False)
                    if len(df.columns) > 1:  # Successfully parsed with multiple columns
                        print(f"  ✅ Successfully loaded with encoding={encoding}, delimiter='{delimiter}'")
                        break
                except Exception:
                    continue
            if df is not None and len(df.columns) > 1:
                break

        if df is None or len(df.columns) <= 1:
            print("  ❌ Failed to load CSV with standard methods, trying pd.read_table...")
            try:
                df = pd.read_table(csv_path, sep=None, engine='python',
                                on_bad_lines='skip', low_memory=False)
                print("  ✅ Successfully loaded with pd.read_table")
            except Exception as e:
                print(f"  ❌ All loading methods failed: {e}")
                return None

        # Store the dataframe
        self.dataframes[name] = df

        # Print summary
        print(f"  📄 Shape: {df.shape}")
        print(f"  🔢 Columns: {', '.join(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")

        return df

    def load_all_csvs(self, directory=CSV_DIR):
        """Load all CSV files in the directory"""
        csv_files = [f for f in os.listdir(directory) if f.lower().endswith('.csv')]

        if not csv_files:
            print("❌ No CSV files found in the directory")
            return

        for csv_file in csv_files:
            csv_path = os.path.join(directory, csv_file)
            self.load_csv(csv_path, csv_file)

        print(f"✅ Loaded {len(csv_files)} CSV files")

    def standardize_particle_columns(self, df=None, name=None):
        """Identify and standardize particle name columns"""
        if df is None:
            if name is None or name not in self.dataframes:
                print("❌ Please provide a valid dataframe or name")
                return None
            df = self.dataframes[name]

        # Clone the dataframe to avoid modifying the original
        standardized_df = df.copy()

        # Look for columns that might contain particle names
        particle_columns = []
        for col in df.columns:
            col_str = str(col).lower()
            if any(term in col_str for term in ['particle', 'name', 'type', 'pdg', 'id']):
                particle_columns.append(col)

        if not particle_columns:
            print("  ℹ️ No obvious particle name columns found")
            return standardized_df

        # For each potential particle column, add standardized versions
        for col in particle_columns:
            if df[col].dtype == object:  # Only process string columns
                # Create new column with canonical names
                new_col_name = f"{col}_canonical"
                standardized_df[new_col_name] = df[col].apply(
                    lambda x: self.alias_manager.get_canonical_name(str(x)) if pd.notna(x) else None
                )

                # Create PDG ID column
                pdg_col_name = f"{col}_pdg_id"
                standardized_df[pdg_col_name] = df[col].apply(
                    lambda x: self.alias_manager.get_pdg_id(str(x)) if pd.notna(x) else None
                )

                print(f"  ✅ Standardized column '{col}' -> '{new_col_name}' and '{pdg_col_name}'")

        return standardized_df

    def standardize_all_dataframes(self):
        """Standardize particle names in all loaded dataframes"""
        standardized_dfs = {}

        for name, df in self.dataframes.items():
            print(f"📊 Standardizing particle names in: {name}")
            standardized_df = self.standardize_particle_columns(df)
            standardized_dfs[f"std_{name}"] = standardized_df

            # Save standardized CSV
            output_path = os.path.join(OUTPUT_DIR, f"standardized_{name}")
            standardized_df.to_csv(output_path, index=False)
            print(f"  💾 Saved standardized version to {output_path}")

        # Add standardized dataframes to the collection
        self.dataframes.update(standardized_dfs)
        print(f"✅ Standardized {len(standardized_dfs)} dataframes")

    def analyze_numerical_columns(self, df=None, name=None):
        """Analyze numerical columns in the dataframe"""
        if df is None:
            if name is None or name not in self.dataframes:
                print("❌ Please provide a valid dataframe or name")
                return None
            df = self.dataframes[name]

        print(f"📊 Analyzing numerical columns...")

        # Get numerical columns
        numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

        if not numerical_cols:
            print("  ℹ️ No numerical columns found")
            return None

        print(f"  🔢 Found {len(numerical_cols)} numerical columns")

        # Basic statistics
        stats_df = df[numerical_cols].describe().T

        # Check for correlations
        if len(numerical_cols) > 1:
            try:
                corr_matrix = df[numerical_cols].corr()

                # Visualize correlation matrix
                plt.figure(figsize=(12, 10))
                mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
                sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
                           annot=True, fmt='.2f', square=True, linewidths=.5)
                plt.title('Correlation Matrix of Numerical Features')
                plt.tight_layout()

                # Save correlation matrix
                if name:
                    output_path = os.path.join(OUTPUT_DIR, f"correlation_matrix_{name.replace('.csv', '')}.png")
                else:
                    output_path = os.path.join(OUTPUT_DIR, "correlation_matrix.png")
                plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.close()

                print(f"  ✅ Saved correlation matrix visualization")
            except Exception as e:
                print(f"  ⚠️ Error creating correlation matrix: {e}")

        # Distribution plots for key columns
        for col in numerical_cols[:5]:  # Limit to first 5 columns
            try:
                plt.figure(figsize=(10, 6))
                sns.histplot(df[col].dropna(), kde=True)
                plt.title(f'Distribution of {col}')
                plt.xlabel(col)
                plt.ylabel('Frequency')

                # Save distribution plot
                if name:
                    output_path = os.path.join(OUTPUT_DIR, f"dist_{col}_{name.replace('.csv', '')}.png")
                else:
                    output_path = os.path.join(OUTPUT_DIR, f"dist_{col}.png")
                plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.close()
            except Exception as e:
                print(f"  ⚠️ Error creating distribution plot for {col}: {e}")

        print(f"  ✅ Created distribution plots for numerical columns")

        # Try PCA if enough numerical columns
        if len(numerical_cols) >= 3:
            try:
                # Standardize data
                scaler = StandardScaler()
                scaled_data = scaler.fit_transform(df[numerical_cols].dropna())

                # Apply PCA
                pca = PCA(n_components=2)
                pca_result = pca.fit_transform(scaled_data)

                # Create PCA plot
                plt.figure(figsize=(10, 8))
                plt.scatter(pca_result[:, 0], pca_result[:, 1], alpha=0.5)
                plt.title('PCA: First Two Principal Components')
                plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
                plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
                plt.grid(True, linestyle='--', alpha=0.7)

                # Save PCA plot
                if name:
                    output_path = os.path.join(OUTPUT_DIR, f"pca_{name.replace('.csv', '')}.png")
                else:
                    output_path = os.path.join(OUTPUT_DIR, "pca_plot.png")
                plt.savefig(output_path, dpi=300, bbox_inches='tight')
                plt.close()

                print(f"  ✅ Created PCA visualization")
            except Exception as e:
                print(f"  ⚠️ Error creating PCA plot: {e}")

        return stats_df

    def analyze_all_dataframes(self):
        """Run analysis on all loaded dataframes"""
        for name, df in self.dataframes.items():
            print(f"📊 Analyzing dataframe: {name}")
            self.analyze_numerical_columns(df, name)

        print(f"✅ Analyzed {len(self.dataframes)} dataframes")

# ====== MAIN PIPELINE ======

def run_full_pipeline():
    # Run the full data processing pipeline
    print("🚀 Starting Advanced Particle Physics Data Processor")
    print("=" * 50)

    # Initialize the particle alias manager
    print("\n📋 Initializing Particle Alias Manager...")
    alias_manager = ParticleAliasManager()

    # Visualize the alias network
    print("\n🔍 Visualizing particle alias relationships...")
    alias_manager.visualize_aliases()

    # Process PDFs
    print("\n📑 Processing PDF files...")
    pdf_processor = PDFProcessor(alias_manager)
    pdf_results = pdf_processor.process_all_pdfs()

    if pdf_results:
        pdf_processor.visualize_particle_mentions(pdf_results)

    # Process CSVs
    print("\n📊 Processing CSV files...")
    csv_analyzer = CSVAnalyzer(alias_manager)
    csv_analyzer.load_all_csvs()

    if csv_analyzer.dataframes:
        csv_analyzer.standardize_all_dataframes()
        csv_analyzer.analyze_all_dataframes()

    print("\n✅ Data processing pipeline completed successfully!")
    print("=" * 50)
    print(f"📁 Output files saved to: {OUTPUT_DIR}")
    print(f"💾 Working files saved to Google Drive: {GDRIVE_PATH}")

# Helper function to upload files to Colab
def upload_files():
    """Upload PDF and CSV files to Colab"""
    from google.colab import files

    print("📤 Please upload your PDF and CSV files...")
    uploaded = files.upload()

    for filename, content in uploaded.items():
        if filename.lower().endswith('.pdf'):
            with open(os.path.join(PDF_DIR, filename), 'wb') as f:
                f.write(content)
            print(f"✅ Saved PDF: {filename}")
        elif filename.lower().endswith('.csv'):
            with open(os.path.join(CSV_DIR, filename), 'wb') as f:
                f.write(content)
            print(f"✅ Saved CSV: {filename}")
        else:
            print(f"⚠️ Unsupported file type: {filename}")

# Interactive mode
def interactive_mode():
    # Run the system in interactive mode
    print("\n🔍 Particle Physics Data Analysis Interactive Mode")
    print("=" * 50)
    print("1. Upload files")
    print("2. Run full pipeline")
    print("3. Process PDFs only")
    print("4. Process CSVs only")
    print("5. Add particle aliases")

In [ ]:
# Generate example signal for PSD analysis
def generate_psd_example():
    """Generate example signal with multiple frequency components"""
    # Parameters
    fs = 1000.0  # Sampling frequency
    duration = 5.0  # Duration in seconds

    # Generate time array
    t = np.arange(0, duration, 1/fs)

    # Generate signal with multiple frequency components
    f1, f2, f3 = 10, 50, 120  # Frequencies in Hz
    signal_data = (
        np.sin(2 * np.pi * f1 * t) +
        0.5 * np.sin(2 * np.pi * f2 * t) +
        0.3 * np.sin(2 * np.pi * f3 * t) +
        0.1 * np.random.randn(len(t))  # Add some noise
    )

    # Plot the signal
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(t, signal_data)
    plt.title('Time Domain Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.grid(True)

    # Compute and plot the PSD
    plt.subplot(2, 1, 2)
    frequencies, psd = compute_psd(signal_data, fs=fs)
    plt.tight_layout()
    plt.show()

    return frequencies, psd, signal_data

# ====================================================================
# PART 9: SHORT-TIME FOURIER TRANSFORM (STFT) / SPECTROGRAM
# ====================================================================

def compute_spectrogram(signal_data, fs=1000.0, n_fft=256, hop_length=128):
    """Compute and display the spectrogram of a signal"""
    frequencies, times, spectrogram = signal.spectrogram(signal_data, fs=fs, nfft=n_fft, noverlap=n_fft - hop_length, mode='magnitude')

    plt.figure(figsize=(10, 6))
    plt.pcolormesh(times, frequencies, 10 * np.log10(spectrogram), shading='auto', cmap='viridis')
    plt.colorbar(label='Power/Frequency (dB)')
    plt.title('Spectrogram')
    plt.xlabel('Time [s]')
    plt.ylabel('Frequency [Hz]')
    plt.tight_layout()
    plt.show()

    return frequencies, times, spectrogram

# Generate example signal for spectrogram analysis (same as PSD)
def generate_spectrogram_example():
    """Generate example signal for spectrogram analysis"""
    fs = 1000.0  # Sampling frequency
    duration = 5.0  # Duration in seconds
    t = np.arange(0, duration, 1/fs)
    f1, f2, f3 = 10, 50, 120  # Frequencies in Hz
    signal_data = (
        np.sin(2 * np.pi * f1 * t) +
        0.5 * np.sin(2 * np.pi * f2 * t) +
        0.3 * np.sin(2 * np.pi * f3 * t) +
        0.1 * np.random.randn(len(t))
    )

    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(t, signal_data)
    plt.title('Time Domain Signal')
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.grid(True)

    plt.subplot(2, 1, 2)
    frequencies, times, spectrogram = compute_spectrogram(signal_data, fs=fs)
    plt.tight_layout()
    plt.show()

    return frequencies, times, spectrogram, signal_data

# ====================================================================
# PART 10: EXAMPLE USAGE AND INTERACTIVE ELEMENTS
# ====================================================================

def run_toolkit():
    model, tokenizer = None, None # Initialize outside the conditional

    def load_mistral_callback(b):
        nonlocal model, tokenizer
        load_button.disabled = True
        status_label.value = "<b style='color:orange'>Loading Mistral 7B...</b>"
        try:
            model, tokenizer = load_language_model()
            status_label.value = "<b style='color:green'>Mistral 7B loaded successfully!</b>"
            prompt_area.disabled = False
            generate_button.disabled = False
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error loading Mistral: {e}</b>"
        finally:
            load_button.disabled = False

    def generate_text_callback(b):
        if model and tokenizer:
            prompt = prompt_area.value
            if prompt.strip():
                generate_button.disabled = True
                status_label.value = "<b style='color:orange'>Generating text...</b>"
                try:
                    response = generate_text(model, tokenizer, prompt,
                                             max_new_tokens=max_tokens_slider.value,
                                             temperature=temp_slider.value,
                                             top_p=top_p_slider.value,
                                             top_k=top_k_slider.value,
                                             repetition_penalty=repetition_penalty_slider.value,
                                             do_sample=do_sample_checkbox.value)
                    output_area.value = response
                    status_label.value = "<b style='color:green'>Text generation complete.</b>"
                except Exception as e:
                    status_label.value = f"<b style='color:red'>Error during generation: {e}</b>"
                finally:
                    generate_button.disabled = False
        else:
            status_label.value = "<b style='color:red'>Mistral 7B not loaded yet. Please click 'Load Mistral 7B'.</b>"

    def run_rnn_example_callback(b):
        status_label.value = "<b style='color:orange'>Running RNN example...</b>"
        try:
            X_train, y_train, _, _ = generate_sine_wave_data()
            input_size = X_train.shape[2]
            hidden_size = 32
            output_size = y_train.shape[1]
            rnn_model = SimpleRNN(input_size, hidden_size, output_size).to(device)
            trained_rnn = train_rnn(rnn_model, X_train, y_train, epochs=5)
            status_label.value = "<b style='color:green'>RNN example complete. Check the plot.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running RNN example: {e}</b>"

    def run_cnn_example_callback(b):
        status_label.value = "<b style='color:orange'>Running CNN example...</b>"
        try:
            train_loader, _ = load_mnist_data()
            cnn_model = SimpleCNN().to(device)
            trained_cnn = train_cnn(cnn_model, train_loader, epochs=1)
            status_label.value = "<b style='color:green'>CNN example complete. Check training output.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running CNN example: {e}</b>"

    def run_gan_example_callback(b):
        status_label.value = "<b style='color:orange'>Running GAN example...</b>"
        try:
            _, train_loader = load_mnist_data(batch_size=128)
            latent_dim = 100
            img_shape = (1, 28, 28)
            generator = Generator(latent_dim, img_shape).to(device)
            discriminator = Discriminator(img_shape).to(device)
            trained_gen, trained_disc = train_gan(generator, discriminator, train_loader, epochs=2, sample_interval=500)
            visualize_gan_output(trained_gen)
            status_label.value = "<b style='color:green'>GAN example complete. Check generated images.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running GAN example: {e}</b>"

    def run_ica_example_callback(b):
        status_label.value = "<b style='color:orange'>Running ICA example...</b>"
        try:
            X, S_ica, ica_model = generate_ica_example()
            status_label.value = "<b style='color:green'>ICA example complete. Check the plots.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running ICA example: {e}</b>"

    def run_pca_example_callback(b):
        status_label.value = "<b style='color:orange'>Running PCA example...</b>"
        try:
            from sklearn.datasets import load_iris
            iris = load_iris()
            X, y = iris.data, iris.target
            visualize_pca(X, y, n_components=2)
            status_label.value = "<b style='color:green'>PCA example complete. Check the plots.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running PCA example: {e}</b>"

    def run_tracking_example_callback(b):
        status_label.value = "<b style='color:orange'>Running Multi-Target Tracking example...</b>"
        try:
            trackers = demonstrate_tracking()
            status_label.value = "<b style='color:green'>Multi-Target Tracking example complete. Check the plot.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running tracking example: {e}</b>"

    def run_psd_example_callback(b):
        status_label.value = "<b style='color:orange'>Running PSD example...</b>"
        try:
            frequencies, psd, signal_data = generate_psd_example()
            status_label.value = "<b style='color:green'>PSD example complete. Check the plots.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running PSD example: {e}</b>"

    def run_spectrogram_example_callback(b):
        status_label.value = "<b style='color:orange'>Running Spectrogram example...</b>"
        try:
            frequencies, times, spectrogram, signal_data = generate_spectrogram_example()
            status_label.value = "<b style='color:green'>Spectrogram example complete. Check the plots.</b>"
        except Exception as e:
            status_label.value = f"<b style='color:red'>Error running Spectrogram example: {e}</b>"

    # Mistral 7B Controls
    load_button = widgets.Button(description="Load Mistral 7B", button_style='primary')
    prompt_area = widgets.Textarea(value='', placeholder='Enter your prompt here...', description='Prompt:', disabled=True, layout=widgets.Layout(width='90%', height='100px'))
    temp_slider = widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.1, description='Temperature:', layout=widgets.Layout(width='50%'))
    top_p_slider = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.1, description='Top P:', layout=widgets.Layout(width='50%'))
    top_k_slider = widgets.IntSlider(value=40, min=1, max=100, step=5, description='Top K:', layout=widgets.Layout(width='50%'))
    repetition_penalty_slider = widgets.FloatSlider(value=1.1, min=1.0, max=2.0, step=0.1, description='Rep. Penalty:', layout=widgets.Layout(width='50%'))
    max_tokens_slider = widgets.IntSlider(value=512, min=64, max=2048, step=64, description='Max Tokens:', layout=widgets.Layout(width='50%'))
    do_sample_checkbox = widgets.Checkbox(value=True, description='Do Sample:', layout=widgets.Layout(width='50%'))
    generate_button = widgets.Button(description="Generate Text", button_style='success', disabled=True)
    output_area = widgets.Textarea(value='', placeholder='Generated text will appear here...', description='Output:', disabled=False, layout=widgets.Layout(width='90%', height='200px'))

    load_button.on_click(load_mistral_callback)
    generate_button.on_click(generate_text_callback)

    mistral_controls = widgets.VBox([
        load_button,
        prompt_area,
        widgets.HBox([temp_slider, top_p_slider]),
        widgets.HBox([top_k_slider, repetition_penalty_slider]),
        max_tokens_slider,
        do_sample_checkbox,
        generate_button,
        output_area
    ])

    # Neural Network Example Buttons
    rnn_button = widgets.Button(description="Run RNN Example", button_style='info')
    cnn_button = widgets.Button(description="Run CNN Example", button_style='info')
    gan_button = widgets.Button(description="Run GAN Example", button_style='info')

    rnn_button.on_click(run_rnn_example_callback)
    cnn_button.on_click(run_cnn_example_callback)
    gan_button.on_click(run_gan_example_callback)

    nn_examples = widgets.HBox([rnn_button, cnn_button, gan_button])

    # Signal Processing Example Buttons
    ica_button = widgets.Button(description="Run ICA Example", button_style='warning')
    pca_button = widgets.Button(description="Run PCA Example", button_style='warning')
    tracking_button = widgets.Button(description="Run Tracking Example", button_style='warning')
    psd_button = widgets.Button(description="Run PSD Example", button_style='warning')
    spectrogram_button = widgets.Button(description="Run Spectrogram Example", button_style='warning')

    ica_button.on_click(run_ica_example_callback)
    pca_button.on_click(run_pca_example_callback)
    tracking_button.on_click(run_tracking_example_callback)
    psd_button.on_click(run_psd_example_callback)
    spectrogram_button.on_click(run_spectrogram_example_callback)

    sp_examples = widgets.HBox([ica_button, pca_button, tracking_button, psd_button, spectrogram_button])

    # Status Label
    status_label = widgets.HTML("Ready")

    # Display the UI
    display(HTML("<h2>Advanced AI Toolkit</h2>"))
    display(HTML("<h3>Mistral 7B Language Model</h3>"))
    display(mistral_controls)
    display(HTML("<h3>Neural Network Examples</h3>"))
    display(nn_examples)
    display(HTML("<h3>Signal Processing Examples</h3>"))
    display(sp_examples)
    display(status_label)

if __name__ == "__main__":
    run_toolkit()

print("\nAI Toolkit interface launched. Interact with the buttons and controls.")

# New Section

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")

messages = [
    {"role": "user", "content": "What is your favourite condiment?"},
    {"role": "assistant", "content": "Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavour to whatever I'm cooking up in the kitchen!"},
    {"role": "user", "content": "Do you have mayonnaise recipes?"}
]

encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

model_inputs = encodeds.to(device)
model.to(device)

generated_ids = model.generate(model_inputs, max_new_tokens=1000, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

In [ ]:
if __name__ == '__main__':
    # Example usage:
    generator = ParticleDataGenerator(seed=123)

    # Customize detector properties
    generator.detector.barrel_radius = 1
    generator.detector.layers = 12
    generator.detector.material_thickness = [0.002, 0.005, 0.009, 0.014, 0.017, 0.021, 0.026]
    generator.detector.b_field = 1.5

    # Customize particle properties
    generator.particle_types['muon'].lifetime = 2.2e-6 * 2 # Increase muon lifetime
    generator.particle_types['strange_quark'] = ParticleProperties(mass=0.095, charge=-1/3, lifetime=1e-10, pdg_id=3)

    # Generate primary vertices with pileup
    pileup_config = {'mean': 5, 'dist': 'poisson'}
    vertices_df = generator.generate_primary_vertex(n_events=5, beam_energy=7000, pileup_model=pileup_config)
    print("Generated Primary Vertices:")
    print(vertices_df.head())

    # Generate particles
    particles_df = generator.generate_particles(vertices_df, particles_per_vertex=5)
    print("\nGenerated Particles:")
    print(particles_df.head())

    # Simulate detector response
    hits_df = generator.simulate_detector_response(particles_df)
    print("\nDetector Hits:")
    print(hits_df.head())

    # Reconstruct tracks
    tracks_df = generator.reconstruct_tracks(hits_df)
    print("\nReconstructed Tracks:")
    print(tracks_df.head())

    # Reconstruct calorimeter clusters
    clusters_df = generator.reconstruct_calorimeter_clusters(hits_df)
    print("\nCalorimeter Clusters:")
    print(clusters_df.head())

    # Find jets
    jets_df = generator.find_jets(clusters_df)
    print("\nReconstructed Jets:")
    print(jets_df.head())

    # Run full simulation and save to CSV
    generator.run_simulation(n_events=5, particles_per_vertex=2, output_dir='simulation_output_csv', output_format='csv')

    # Visualize the first event
    if not vertices_df.empty and not particles_df.empty and not hits_df.empty and not tracks_df.empty:
        generator.visualize_event(vertices_df, particles_df, hits_df, tracks_df, event_id=vertices_df['event_id'].iloc[0])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import expon, norm, crystalball, gamma, pareto
from scipy.interpolate import interp1d
import os
from tqdm.notebook import tqdm
import pickle
import time
import warnings
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable, Any
from matplotlib.patches import Circle, Rectangle
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import multiprocessing as mp
from functools import partial
import json
import h5py

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('ParticleDataGenerator')


@dataclass
class ParticleProperties:
    """Data class to store particle physics properties"""
    mass: float
    charge: float
    lifetime: float
    width: float = 0.0  # Natural width in GeV
    color_charge: bool = False
    spin: float = 0.0
    pdg_id: int = 0
    decay_modes: Dict[str, float] = field(default_factory=dict)

    def __post_init__(self):
        # Validate properties
        assert self.mass >= 0, f"Mass must be non-negative: {self.mass}"
        assert isinstance(self.charge, (int, float)), f"Charge must be numeric: {self.charge}"
        assert self.lifetime >= 0, f"Lifetime must be non-negative: {self.lifetime}"


@dataclass
class DetectorProperties:
    """Data class to store detector properties"""
    # Geometry
    barrel_radius: float = 1.1  # meters
    barrel_length: float = 5.0  # meters
    endcap_distance: float = 2.5  # meters from center
    layers: int = 5  # number of detector layers

    # Material budget in radiation lengths per layer
    material_thickness: List[float] = field(default_factory=lambda: [0.01, 0.03, 0.05, 0.10, 0.15])

    # Magnetic field
    b_field: float = 2.0  # Tesla
    b_field_direction: str = 'z'  # Direction of magnetic field

    # Resolution properties
    position_resolution: Dict[str, float] = field(
        default_factory=lambda: {'barrel': 0.01, 'endcap': 0.02}  # meters
    )
    energy_resolution: Dict[str, Dict[str, float]] = field(
        default_factory=lambda: {
            'em_calorimeter': {'stochastic': 0.10, 'constant': 0.01, 'noise': 0.1},  # EM calorimeter
            'had_calorimeter': {'stochastic': 0.50, 'constant': 0.03, 'noise': 0.5}   # Hadronic calorimeter
        }
    )
    time_resolution: float = 0.15e-9  # seconds (150 ps)

    # Tracker properties
    tracker_efficiency: Dict[str, float] = field(
        default_factory=lambda: {'electron': 0.99, 'muon': 0.98, 'pion': 0.95, 'kaon': 0.94, 'proton': 0.93}
    )

    # Calorimeter properties
    calorimeter_acceptance: Dict[str, float] = field(
        default_factory=lambda: {'eta_min': -3.0, 'eta_max': 3.0}
    )

    # Trigger thresholds
    trigger_thresholds: Dict[str, float] = field(
        default_factory=lambda: {'electron_pt': 20.0, 'muon_pt': 15.0, 'jet_pt': 30.0, 'met': 50.0}
    )

    def __post_init__(self):
        """Validate detector properties"""
        assert self.barrel_radius > 0, f"Barrel radius must be positive: {self.barrel_radius}"
        assert self.barrel_length > 0, f"Barrel length must be positive: {self.barrel_length}"
        assert len(self.material_thickness) == self.layers, (
            f"Material thickness list length ({len(self.material_thickness)}) "
            f"must match number of layers ({self.layers})"
        )


class PhysicsModels:
    """
    Class containing physics models for particle generation and interactions
    """
    @staticmethod
    def bjorken_x_distribution(x: np.ndarray, alpha: float = 0.5, beta: float = 3.0) -> np.ndarray:
        """Compute parton distribution function (simplified)"""
        return np.power(x, -alpha) * np.power(1-x, beta)

    @staticmethod
    def pt_spectrum(pt: np.ndarray, p0: float = 2.0, n: float = 5.0) -> np.ndarray:
        """Compute pT spectrum using power law distribution"""
        return np.power(1.0 + pt/p0, -n)

    @staticmethod
    def crystal_ball_mass(m: np.ndarray, m0: float, sigma: float,
                         alpha: float = 1.0, n: float = 3.0) -> np.ndarray:
        """Crystal Ball function for resonance mass distribution"""
        return crystalball.pdf(m, alpha, n, loc=m0, scale=sigma)

    @staticmethod
    def breit_wigner(m: np.ndarray, m0: float, gamma: float) -> np.ndarray:
        """Relativistic Breit-Wigner distribution for resonance mass"""
        return gamma / (2 * np.pi * ((m - m0)**2 + (gamma/2)**2))

    @staticmethod
    def fragmentation_function(z: np.ndarray, a: float = 0.3, b: float = 0.58) -> np.ndarray:
        """Fragmentation function for quark hadronization"""
        return (1/z) * np.power(1-z, a) * np.exp(-b * np.square(1/z))

    @staticmethod
    def decay_probability(lifetime: float, proper_time: float) -> float:
        """Calculate decay probability based on particle lifetime"""
        if lifetime == np.inf:
            return 0.0

        # Exponential decay law
        return 1.0 - np.exp(-proper_time / lifetime)


class ParticleDataGenerator:
    """
    Enhanced generator for synthetic particle physics data with configurable properties.
    """

    def __init__(self, detector_config: Optional[Dict] = None,
                 particle_config: Optional[Dict] = None, seed: int = 42):
        """Initialize the particle data generator"""
        # Set random seed
        np.random.seed(seed)
        self.seed = seed

        # Initialize physics models
        self.physics = PhysicsModels()

        # Initialize particle properties
        self._init_particle_properties(particle_config)

        # Initialize detector properties
        self._init_detector_properties(detector_config)

        # Physics constants
        self.constants = {
            'c': 299792458,   # Speed of light in m/s
            'hbar': 6.582e-25, # GeV·s
            'alpha_em': 1/137, # Fine structure constant
            'G_F': 1.166e-5,   # Fermi constant in GeV^-2
        }

        # Track counters for unique IDs
        self.event_counter = 0
        self.vertex_counter = 0
        self.particle_counter = 0
        self.hit_counter = 0
        self.jet_counter = 0

        # Logging setup
        self.logger = logging.getLogger(__name__)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import expon, norm, crystalball, gamma, pareto
from scipy.interpolate import interp1d
import os
from tqdm.notebook import tqdm
import pickle
import time
import warnings
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable, Any
from matplotlib.patches import Circle, Rectangle
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import multiprocessing as mp
from functools import partial
import json
import h5py

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('ParticleDataGenerator')


@dataclass
class ParticleProperties:
    """Data class to store particle physics properties"""
    mass: float
    charge: float
    lifetime: float
    width: float = 0.0  # Natural width in GeV
    color_charge: bool = False
    spin: float = 0.0
    pdg_id: int = 0
    decay_modes: Dict[str, float] = field(default_factory=dict)

    def __post_init__(self):
        # Validate properties
        assert self.mass >= 0, f"Mass must be non-negative: {self.mass}"
        assert isinstance(self.charge, (int, float)), f"Charge must be numeric: {self.charge}"
        assert self.lifetime >= 0, f"Lifetime must be non-negative: {self.lifetime}"


@dataclass
class DetectorProperties:
    """Data class to store detector properties"""
    # Geometry
    barrel_radius: float = 1.1  # meters
    barrel_length: float = 5.0  # meters
    endcap_distance: float = 2.5  # meters from center
    layers: int = 5  # number of detector layers

    # Material budget in radiation lengths per layer
    material_thickness: List[float] = field(default_factory=lambda: [0.01, 0.03, 0.05, 0.10, 0.15])

    # Magnetic field
    b_field: float = 2.0  # Tesla
    b_field_direction: str = 'z'  # Direction of magnetic field

    # Resolution properties
    position_resolution: Dict[str, float] = field(
        default_factory=lambda: {'barrel': 0.01, 'endcap': 0.02}  # meters
    )
    energy_resolution: Dict[str, Dict[str, float]] = field(
        default_factory=lambda: {
            'em_calorimeter': {'stochastic': 0.10, 'constant': 0.01, 'noise': 0.1},  # EM calorimeter
            'had_calorimeter': {'stochastic': 0.50, 'constant': 0.03, 'noise': 0.5}   # Hadronic calorimeter
        }
    )
    time_resolution: float = 0.15e-9  # seconds (150 ps)

    # Tracker properties
    tracker_efficiency: Dict[str, float] = field(
        default_factory=lambda: {'electron': 0.99, 'muon': 0.98, 'pion': 0.95, 'kaon': 0.94, 'proton': 0.93}
    )

    # Calorimeter properties
    calorimeter_acceptance: Dict[str, float] = field(
        default_factory=lambda: {'eta_min': -3.0, 'eta_max': 3.0}
    )

    # Trigger thresholds
    trigger_thresholds: Dict[str, float] = field(
        default_factory=lambda: {'electron_pt': 20.0, 'muon_pt': 15.0, 'jet_pt': 30.0, 'met': 50.0}
    )

    def __post_init__(self):
        """Validate detector properties"""
        assert self.barrel_radius > 0, f"Barrel radius must be positive: {self.barrel_radius}"
        assert self.barrel_length > 0, f"Barrel length must be positive: {self.barrel_length}"
        assert len(self.material_thickness) == self.layers, (
            f"Material thickness list length ({len(self.material_thickness)}) "
            f"must match number of layers ({self.layers})"
        )


class PhysicsModels:
    """
    Class containing physics models for particle generation and interactions
    """
    @staticmethod
    def bjorken_x_distribution(x: np.ndarray, alpha: float = 0.5, beta: float = 3.0) -> np.ndarray:
        """Compute parton distribution function (simplified)"""
        return np.power(x, -alpha) * np.power(1-x, beta)

    @staticmethod
    def pt_spectrum(pt: np.ndarray, p0: float = 2.0, n: float = 5.0) -> np.ndarray:
        """Compute pT spectrum using power law distribution"""
        return np.power(1.0 + pt/p0, -n)

    @staticmethod
    def crystal_ball_mass(m: np.ndarray, m0: float, sigma: float,
                         alpha: float = 1.0, n: float = 3.0) -> np.ndarray:
        """Crystal Ball function for resonance mass distribution"""
        return crystalball.pdf(m, alpha, n, loc=m0, scale=sigma)

    @staticmethod
    def breit_wigner(m: np.ndarray, m0: float, gamma: float) -> np.ndarray:
        """Relativistic Breit-Wigner distribution for resonance mass"""
        return gamma / (2 * np.pi * ((m - m0)**2 + (gamma/2)**2))

    @staticmethod
    def fragmentation_function(z: np.ndarray, a: float = 0.3, b: float = 0.58) -> np.ndarray:
        """Fragmentation function for quark hadronization"""
        return (1/z) * np.power(1-z, a) * np.exp(-b * np.square(1/z))

    @staticmethod
    def decay_probability(lifetime: float, proper_time: float) -> float:
        """Calculate decay probability based on particle lifetime"""
        if lifetime == np.inf:
            return 0.0

        # Exponential decay law
        return 1.0 - np.exp(-proper_time / lifetime)


class ParticleDataGenerator:
    """
    Enhanced generator for synthetic particle physics data with configurable properties.
    """

    def __init__(self, detector_config: Optional[Dict] = None,
                 particle_config: Optional[Dict] = None, seed: int = 42):
        """Initialize the particle data generator"""
        # Set random seed
        np.random.seed(seed)
        self.seed = seed

        # Initialize physics models
        self.physics = PhysicsModels()

        # Initialize particle properties
        self._init_particle_properties(particle_config)

        # Initialize detector properties
        self._init_detector_properties(detector_config)

        # Physics constants
        self.constants = {
            'c': 299792458,   # Speed of light in m/s
            'hbar': 6.582e-25, # GeV·s
            'alpha_em': 1/137, # Fine structure constant
            'G_F': 1.166e-5,   # Fermi constant in GeV^-2
        }

        # Track counters for unique IDs
        self.event_counter = 0
        self.vertex_counter = 0
        self.particle_counter = 0
        self.hit_counter = 0
        self.jet_counter = 0

        # Logging setup
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"ParticleDataGenerator initialized with seed {seed}")

    def _init_particle_properties(self, particle_config: Optional[Dict] = None):
        """Initialize particle properties"""
        pass # Replace with actual implementation

    def _init_detector_properties(self, detector_config: Optional[Dict] = None):
        """Initialize detector properties"""
        pass # Replace with actual implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import expon, norm, crystalball, gamma, pareto
from scipy.interpolate import interp1d
import os
from tqdm.notebook import tqdm
import pickle
import time
import warnings
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable, Any
from matplotlib.patches import Circle, Rectangle
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import multiprocessing as mp
from functools import partial
import json
import h5py

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('ParticleDataGenerator')


@dataclass
class ParticleProperties:
    """Data class to store particle physics properties"""
    mass: float
    charge: float
    lifetime: float
    width: float = 0.0  # Natural width in GeV
    color_charge: bool = False
    spin: float = 0.0
    pdg_id: int = 0
    decay_modes: Dict[str, float] = field(default_factory=dict)

    def __post_init__(self):
        # Validate properties
        assert self.mass >= 0, f"Mass must be non-negative: {self.mass}"
        assert isinstance(self.charge, (int, float)), f"Charge must be numeric: {self.charge}"
        assert self.lifetime >= 0, f"Lifetime must be non-negative: {self.lifetime}"


@dataclass
class DetectorProperties:
    """Data class to store detector properties"""
    # Geometry
    barrel_radius: float = 1.1  # meters
    barrel_length: float = 5.0  # meters
    endcap_distance: float = 2.5  # meters from center
    layers: int = 5  # number of detector layers

    # Material budget in radiation lengths per layer
    material_thickness: List[float] = field(default_factory=lambda: [0.01, 0.03, 0.05, 0.10, 0.15])

    # Magnetic field
    b_field: float = 2.0  # Tesla
    b_field_direction: str = 'z'  # Direction of magnetic field

    # Resolution properties
    position_resolution: Dict[str, float] = field(
        default_factory=lambda: {'barrel': 0.01, 'endcap': 0.02}  # meters
    )
    energy_resolution: Dict[str, Dict[str, float]] = field(
        default_factory=lambda: {
            'em_calorimeter': {'stochastic': 0.10, 'constant': 0.01, 'noise': 0.1},  # EM calorimeter
            'had_calorimeter': {'stochastic': 0.50, 'constant': 0.03, 'noise': 0.5}   # Hadronic calorimeter
        }
    )
    time_resolution: float = 0.15e-9  # seconds (150 ps)

    # Tracker properties
    tracker_efficiency: Dict[str, float] = field(
        default_factory=lambda: {'electron': 0.99, 'muon': 0.98, 'pion': 0.95, 'kaon': 0.94, 'proton': 0.93}
    )

    # Calorimeter properties
    calorimeter_acceptance: Dict[str, float] = field(
        default_factory=lambda: {'eta_min': -3.0, 'eta_max': 3.0}
    )

    # Trigger thresholds
    trigger_thresholds: Dict[str, float] = field(
        default_factory=lambda: {'electron_pt': 20.0, 'muon_pt': 15.0, 'jet_pt': 30.0, 'met': 50.0}
    )

    def __post_init__(self):
        """Validate detector properties"""
        assert self.barrel_radius > 0, f"Barrel radius must be positive: {self.barrel_radius}"
        assert self.barrel_length > 0, f"Barrel length must be positive: {self.barrel_length}"
        assert len(self.material_thickness) == self.layers, (
            f"Material thickness list length ({len(self.material_thickness)}) "
            f"must match number of layers ({self.layers})"
        )


class PhysicsModels:
    """
    Class containing physics models for particle generation and interactions
    """
    @staticmethod
    def bjorken_x_distribution(x: np.ndarray, alpha: float = 0.5, beta: float = 3.0) -> np.ndarray:
        """Compute parton distribution function (simplified)"""
        return np.power(x, -alpha) * np.power(1-x, beta)

    @staticmethod
    def pt_spectrum(pt: np.ndarray, p0: float = 2.0, n: float = 5.0) -> np.ndarray:
        """Compute pT spectrum using power law distribution"""
        return np.power(1.0 + pt/p0, -n)

    @staticmethod
    def crystal_ball_mass(m: np.ndarray, m0: float, sigma: float,
                         alpha: float = 1.0, n: float = 3.0) -> np.ndarray:
        """Crystal Ball function for resonance mass distribution"""
        return crystalball.pdf(m, alpha, n, loc=m0, scale=sigma)

    @staticmethod
    def breit_wigner(m: np.ndarray, m0: float, gamma: float) -> np.ndarray:
        """Relativistic Breit-Wigner distribution for resonance mass"""
        return gamma / (2 * np.pi * ((m - m0)**2 + (gamma/2)**2))

    @staticmethod
    def fragmentation_function(z: np.ndarray, a: float = 0.3, b: float = 0.58) -> np.ndarray:
        """Fragmentation function for quark hadronization"""
        return (1/z) * np.power(1-z, a) * np.exp(-b * np.square(1/z))

    @staticmethod
    def decay_probability(lifetime: float, proper_time: float) -> float:
        """Calculate decay probability based on particle lifetime"""
        if lifetime == np.inf:
            return 0.0

        # Exponential decay law
        return 1.0 - np.exp(-proper_time / lifetime)


class ParticleDataGenerator:
    """
    Enhanced generator for synthetic particle physics data with configurable properties.
    """

    def __init__(self, detector_config: Optional[Dict] = None,
                 particle_config: Optional[Dict] = None, seed: int = 42):
        """Initialize the particle data generator"""
        # Set random seed
        np.random.seed(seed)
        self.seed = seed

        # Initialize physics models
        self.physics = PhysicsModels()

        # Initialize particle properties
        self._init_particle_properties(particle_config)

        # Initialize detector properties
        self._init_detector_properties(detector_config)

        # Physics constants
        self.constants = {
            'c': 299792458,   # Speed of light in m/s
            'hbar': 6.582e-25, # GeV·s
            'alpha_em': 1/137, # Fine structure constant
            'G_F': 1.166e-5,   # Fermi constant in GeV^-2
        }

        # Track counters for unique IDs
        self.event_counter = 0
        self.vertex_counter = 0
        self.particle_counter = 0
        self.hit_counter = 0
        self.jet_counter = 0

        # Logging setup
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"ParticleDataGenerator initialized with seed {seed}")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import expon, norm, crystalball, gamma, pareto
from scipy.interpolate import interp1d
import os
from tqdm.notebook import tqdm
import pickle
import time
import warnings
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable, Any
from matplotlib.patches import Circle, Rectangle
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import multiprocessing as mp
from functools import partial
import json
import h5py

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('ParticleDataGenerator')


@dataclass
class ParticleProperties:
    """Data class to store particle physics properties"""
    mass: float
    charge: float
    lifetime: float
    width: float = 0.0  # Natural width in GeV
    color_charge: bool = False
    spin: float = 0.0
    pdg_id: int = 0
    decay_modes: Dict[str, float] = field(default_factory=dict)

    def __post_init__(self):
        # Validate properties
        assert self.mass >= 0, f"Mass must be non-negative: {self.mass}"
        assert isinstance(self.charge, (int, float)), f"Charge must be numeric: {self.charge}"
        assert self.lifetime >= 0, f"Lifetime must be non-negative: {self.lifetime}"


@dataclass
class DetectorProperties:
    """Data class to store detector properties"""
    # Geometry
    barrel_radius: float = 1.1  # meters
    barrel_length: float = 5.0  # meters
    endcap_distance: float = 2.5  # meters from center
    layers: int = 5  # number of detector layers

    # Material budget in radiation lengths per layer
    material_thickness: List[float] = field(default_factory=lambda: [0.01, 0.03, 0.05, 0.10, 0.15])

    # Magnetic field
    b_field: float = 2.0  # Tesla
    b_field_direction: str = 'z'  # Direction of magnetic field

    # Resolution properties
    position_resolution: Dict[str, float] = field(
        default_factory=lambda: {'barrel': 0.01, 'endcap': 0.02}  # meters
    )
    energy_resolution: Dict[str, Dict[str, float]] = field(
        default_factory=lambda: {
            'em_calorimeter': {'stochastic': 0.10, 'constant': 0.01, 'noise': 0.1},  # EM calorimeter
            'had_calorimeter': {'stochastic': 0.50, 'constant': 0.03, 'noise': 0.5}   # Hadronic calorimeter
        }
    )
    time_resolution: float = 0.15e-9  # seconds (150 ps)

    # Tracker properties
    tracker_efficiency: Dict[str, float] = field(
        default_factory=lambda: {'electron': 0.99, 'muon': 0.98, 'pion': 0.95, 'kaon': 0.94, 'proton': 0.93}
    )

    # Calorimeter properties
    calorimeter_acceptance: Dict[str, float] = field(
        default_factory=lambda: {'eta_min': -3.0, 'eta_max': 3.0}
    )

    # Trigger thresholds
    trigger_thresholds: Dict[str, float] = field(
        default_factory=lambda: {'electron_pt': 20.0, 'muon_pt': 15.0, 'jet_pt': 30.0, 'met': 50.0}
    )

    def __post_init__(self):
        """Validate detector properties"""
        assert self.barrel_radius > 0, f"Barrel radius must be positive: {self.barrel_radius}"
        assert self.barrel_length > 0, f"Barrel length must be positive: {self.barrel_length}"
        assert len(self.material_thickness) == self.layers, (
            f"Material thickness list length ({len(self.material_thickness)}) "
            f"must match number of layers ({self.layers})"
        )


class PhysicsModels:
    """
    Class containing physics models for particle generation and interactions
    """
    @staticmethod
    def bjorken_x_distribution(x: np.ndarray, alpha: float = 0.5, beta: float = 3.0) -> np.ndarray:
        """Compute parton distribution function (simplified)"""
        return np.power(x, -alpha) * np.power(1-x, beta)

    @staticmethod
    def pt_spectrum(pt: np.ndarray, p0: float = 2.0, n: float = 5.0) -> np.ndarray:
        """Compute pT spectrum using power law distribution"""
        return np.power(1.0 + pt/p0, -n)

    @staticmethod
    def crystal_ball_mass(m: np.ndarray, m0: float, sigma: float,
                         alpha: float = 1.0, n: float = 3.0) -> np.ndarray:
        """Crystal Ball function for resonance mass distribution"""
        return crystalball.pdf(m, alpha, n, loc=m0, scale=sigma)

    @staticmethod
    def breit_wigner(m: np.ndarray, m0: float, gamma: float) -> np.ndarray:
        """Relativistic Breit-Wigner distribution for resonance mass"""
        return gamma / (2 * np.pi * ((m - m0)**2 + (gamma/2)**2))

    @staticmethod
    def fragmentation_function(z: np.ndarray, a: float = 0.3, b: float = 0.58) -> np.ndarray:
        """Fragmentation function for quark hadronization"""
        return (1/z) * np.power(1-z, a) * np.exp(-b * np.square(1/z))

    @staticmethod
    def decay_probability(lifetime: float, proper_time: float) -> float:
        """Calculate decay probability based on particle lifetime"""
        if lifetime == np.inf:
            return 0.0

        # Exponential decay law
        return 1.0 - np.exp(-proper_time / lifetime)


class ParticleDataGenerator:
    """
    Enhanced generator for synthetic particle physics data with configurable properties.
    """

    def __init__(self, detector_config: Optional[Dict] = None,
                 particle_config: Optional[Dict] = None, seed: int = 42):
        """Initialize the particle data generator"""
        # Set random seed
        np.random.seed(seed)
        self.seed = seed

        # Initialize physics models
        self.physics = PhysicsModels()

        # Initialize particle properties
        self._init_particle_properties(particle_config)

        # Initialize detector properties
        self._init_detector_properties(detector_config)

        # Physics constants
        self.constants = {
            'c': 299792458,   # Speed of light in m/s
            'hbar': 6.582e-25, # GeV·s
            'alpha_em': 1/137, # Fine structure constant
            'G_F': 1.166e-5,   # Fermi constant in GeV^-2
        }

        # Track counters for unique IDs
        self.event_counter = 0
        self.vertex_counter = 0
        self.particle_counter = 0
        self.hit_counter = 0
        self.jet_counter = 0

        # Logging setup
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"ParticleDataGenerator initialized with seed {seed}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import expon, norm, crystalball
import os
from tqdm.notebook import tqdm
import pickle
import time

class ParticleDataGenerator:
    """
    Generates synthetic particle physics data with configurable properties.

    This generator creates realistic particle data with properties like:
    - Energy distributions
    - Momentum vectors
    - Decay chains
    - Interaction probabilities
    - Detector response simulation
    """

    def __init__(self, seed=42):
        """Initialize the particle data generator with optional seed for reproducibility"""
        np.random.seed(seed)
        self.particle_types = {
            'electron': {'mass': 0.000511, 'charge': -1, 'lifetime': np.inf},
            'muon': {'mass': 0.1057, 'charge': -1, 'lifetime': 2.2e-6},
            'photon': {'mass': 0, 'charge': 0, 'lifetime': np.inf},
            'proton': {'mass': 0.93827, 'charge': 1, 'lifetime': np.inf},
            'neutron': {'mass': 0.93957, 'charge': 0, 'lifetime': 880},
            'pion': {'mass': 0.13957, 'charge': 1, 'lifetime': 2.6e-8},
            'kaon': {'mass': 0.49368, 'charge': 1, 'lifetime': 1.24e-8},
            'tau': {'mass': 1.7768, 'charge': -1, 'lifetime': 2.9e-13},
            'Higgs': {'mass': 125.0, 'charge': 0, 'lifetime': 1.56e-22},
            'Z': {'mass': 91.188, 'charge': 0, 'lifetime': 2.65e-25},
        }

        # Define detector properties
        self.detector_dimensions = {
            'barrel_radius': 1.1,    # meters
            'barrel_length': 5.0,    # meters
            'endcap_distance': 2.5,  # meters from center
            'layers': 5,             # number of detector layers
            'resolution': {
                'position': 0.01,    # meters
                'energy': 0.03,      # relative resolution (3%)
                'time': 0.15e-9      # seconds (150 ps)
            }
        }

    def generate_primary_vertex(self, n_events=1000, beam_energy=6500):
        """Generate primary interaction vertices for collision events"""
        vertices = []

        # Generate vertex positions - beam collision region
        x = np.random.normal(0, 0.015, n_events)  # 15 μm in x
        y = np.random.normal(0, 0.015, n_events)  # 15 μm in y
        z = np.random.normal(0, 5.5, n_events)    # 5.5 cm in z

        # Generate event times (bunch crossing structure)
        t = np.random.uniform(0, 25e-9, n_events)  # 25 ns bunch spacing

        # Track multiplicity follows negative binomial distribution
        k, p = 1.5, 0.05  # parameters for negative binomial
        mult = np.random.negative_binomial(k, p, n_events) + 5  # add minimum multiplicity

        # Collision energy - can fluctuate around beam energy
        energy = np.random.normal(beam_energy, beam_energy*0.05, n_events)

        for i in range(n_events):
            vertices.append({
                'event_id': i,
                'x': x[i],
                'y': y[i],
                'z': z[i],
                't': t[i],
                'multiplicity': mult[i],
                'energy': energy[i]
            })

        return pd.DataFrame(vertices)

    def _sample_pt_distribution(self, n_particles, mean_pt=1.2, max_pt=100):
        """Sample from realistic pT distribution (~ exp(-pT/mean_pT))"""
        return expon.rvs(scale=mean_pt, size=n_particles).clip(0.1, max_pt)

    def _sample_eta_distribution(self, n_particles, eta_max=2.5):
        """Sample pseudorapidity distribution"""
        return np.random.uniform(-eta_max, eta_max, n_particles)

    def _sample_phi_distribution(self, n_particles):
        """Sample uniform phi distribution"""
        return np.random.uniform(-np.pi, np.pi, n_particles)

    def _pt_eta_phi_to_xyz(self, pt, eta, phi):
        """Convert pT, eta, phi to px, py, pz"""
        px = pt * np.cos(phi)
        py = pt * np.sin(phi)
        pz = pt * np.sinh(eta)
        return px, py, pz

    def generate_particles(self, vertices_df, particle_types=None, pt_mean=1.2):
        """Generate particles from collision vertices"""
        if particle_types is None:
            # Default particle type probabilities
            particle_types = {
                'electron': 0.05,
                'muon': 0.05,
                'photon': 0.20,
                'pion': 0.50,
                'kaon': 0.12,
                'proton': 0.08
            }

        particles = []
        particle_id = 0

        for _, vertex in tqdm(vertices_df.iterrows(), total=len(vertices_df), desc="Generating particles"):
            # How many particles in this event
            n_particles = vertex['multiplicity']

            # Sample kinematics
            pt = self._sample_pt_distribution(n_particles, mean_pt=pt_mean)
            eta = self._sample_eta_distribution(n_particles)
            phi = self._sample_phi_distribution(n_particles)

            # Sample particle types based on provided probabilities
            types = np.random.choice(
                list(particle_types.keys()),
                size=n_particles,
                p=list(particle_types.values())
            )

            # Generate each particle
            for i in range(n_particles):
                px, py, pz = self._pt_eta_phi_to_xyz(pt[i], eta[i], phi[i])

                # Get particle properties
                p_type = types[i]
                mass = self.particle_types[p_type]['mass']
                charge = self.particle_types[p_type]['charge']

                # Calculate energy: E² = p² + m²
                p_squared = px**2 + py**2 + pz**2
                energy = np.sqrt(p_squared + mass**2)

                # Add random variation to vertex position for this particle
                dx = np.random.normal(0, 0.0001)  # 100 μm variation
                dy = np.random.normal(0, 0.0001)
                dz = np.random.normal(0, 0.0001)

                particles.append({
                    'particle_id': particle_id,
                    'event_id': vertex['event_id'],
                    'type': p_type,
                    'mass': mass,
                    'charge': charge,
                    'px': px,
                    'py': py,
                    'pz': pz,
                    'pt': pt[i],
                    'eta': eta[i],
                    'phi': phi[i],
                    'energy': energy,
                    'creation_x': vertex['x'] + dx,
                    'creation_y': vertex['y'] + dy,
                    'creation_z': vertex['z'] + dz,
                    'creation_t': vertex['t'],
                })
                particle_id += 1

        return pd.DataFrame(particles)

    def simulate_detector_hits(self, particles_df, efficiency=0.95, noise_rate=1e-6):
        """Simulate detector response to particles"""
        hits = []
        hit_id = 0

        # Calculate number of noise hits based on detector volume and noise rate
        detector_volume = (
            np.pi * self.detector_dimensions['barrel_radius']**2 *
            self.detector_dimensions['barrel_length'] *
            self.detector_dimensions['layers']
        )
        n_noise_hits = int(detector_volume * noise_rate * len(particles_df['event_id'].unique()))

        # Process each particle
        for _, particle in tqdm(particles_df.iterrows(), total=len(particles_df), desc="Simulating hits"):
            # Check if particle creates hits (based on efficiency)
            if np.random.random() > efficiency:
                continue

            # Get initial position and momentum
            x0, y0, z0 = particle['creation_x'], particle['creation_y'], particle['creation_z']
            px, py, pz = particle['px'], particle['py'], particle['pz']
            p_mag = np.sqrt(px**2 + py**2 + pz**2)

            # Unit vector in momentum direction
            if p_mag > 0:
                dx, dy, dz = px/p_mag, py/p_mag, pz/p_mag
            else:
                continue  # Skip particles with zero momentum

            # Particle parameters
            charge = particle['charge']
            mass = particle['mass']
            energy = particle['energy']

            # Skip neutral particles for tracking detector
            if charge == 0:
                # For neutral particles, just simulate calorimeter deposit
                # Distance to calorimeter (assume fixed radius)
                calorimeter_radius = self.detector_dimensions['barrel_radius'] * 1.1

                # Simple straight line to calorimeter
                r_xy = np.sqrt(x0**2 + y0**2)

                if r_xy < 1e-10:  # Avoid division by zero
                    t_hit = 0
                else:
                    # Time to reach calorimeter
                    t_hit = (calorimeter_radius - r_xy) * p_mag / energy

                if t_hit > 0:
                    # Position at calorimeter
                    x_hit = x0 + dx * calorimeter_radius
                    y_hit = y0 + dy * calorimeter_radius
                    z_hit = z0 + dz * calorimeter_radius
                    t_hit = particle['creation_t'] + t_hit

                    # Energy deposit with resolution
                    energy_smeared = np.random.normal(
                        energy,
                        energy * self.detector_dimensions['resolution']['energy']
                    )

                    hits.append({
                        'hit_id': hit_id,
                        'event_id': particle['event_id'],
                        'particle_id': particle['particle_id'],
                        'detector': 'calorimeter',
                        'layer': 0,
                        'x': x_hit,
                        'y': y_hit,
                        'z': z_hit,
                        't': t_hit,
                        'energy': energy_smeared,
                        'is_noise': False
                    })
                    hit_id += 1

            else:
                # For charged particles, simulate trajectory with detector hits
                # Simplified helix trajectory in magnetic field (assume 2T along z)
                B = 2.0  # Tesla

                # Radius of curvature in xy-plane
                # R[m] = p_T[GeV] / (0.3 * B[T])
                pt = np.sqrt(px**2 + py**2)
                R = pt / (0.3 * B) if pt > 0 else float('inf')

                # Generate hits in detector layers
                for layer in range(self.detector_dimensions['layers']):
                    # Simplified hit position calculation
                    layer_radius = (layer + 1) * self.detector_dimensions['barrel_radius'] / self.detector_dimensions['layers']

                    # Simple helix parametrization (very simplified)
                    phi0 = np.arctan2(py, px)

                    if R > layer_radius:  # Particle reaches this layer
                        # Arc length to reach this radius
                        arc_length = layer_radius

                        # Position at this radius (simplified)
                        phi_hit = phi0 + arc_length / R if R != float('inf') else phi0
                        x_hit = layer_radius * np.cos(phi_hit)
                        y_hit = layer_radius * np.sin(phi_hit)

                        # z position based on pz component
                        z_hit = z0 + dz * arc_length

                        # Time to reach this layer
                        v = p_mag / np.sqrt(energy**2 - mass**2) if energy > mass else 0
                        t_hit = particle['creation_t'] + arc_length / v if v > 0 else float('inf')

                        # Apply detector resolution
                        x_hit += np.random.normal(0, self.detector_dimensions['resolution']['position'])
                        y_hit += np.random.normal(0, self.detector_dimensions['resolution']['position'])
                        z_hit += np.random.normal(0, self.detector_dimensions['resolution']['position'])
                        t_hit += np.random.normal(0, self.detector_dimensions['resolution']['time'])

                        # Smaller energy deposits in tracking layers
                        tracking_deposit = np.random.gamma(2.0, 0.001)

                        hits.append({
                            'hit_id': hit_id,
                            'event_id': particle['event_id'],
                            'particle_id': particle['particle_id'],
                            'detector': 'tracker',
                            'layer': layer,
                            'x': x_hit,
                            'y': y_hit,
                            'z': z_hit,
                            't': t_hit,
                            'energy': tracking_deposit,
                            'is_noise': False
                        })
                        hit_id += 1

        # Generate random noise hits
        if n_noise_hits > 0:
            noise_events = np.random.choice(particles_df['event_id'].unique(), n_noise_hits)

            for i in range(n_noise_hits):
                # Random position within detector volume
                r = np.random.uniform(0, self.detector_dimensions['barrel_radius'])
                phi = np.random.uniform(0, 2*np.pi)
                z = np.random.uniform(-self.detector_dimensions['barrel_length']/2,
                                     self.detector_dimensions['barrel_length']/2)

                x_noise = r * np.cos(phi)
                y_noise = r * np.sin(phi)

                # Random detector layer
                layer = np.random.randint(0, self.detector_dimensions['layers'])

                # Random time within event window
                t_noise = np.random.uniform(0, 25e-9)  # 25ns window

                # Small random energy deposit
                e_noise = np.random.exponential(0.001)

                hits.append({
                    'hit_id': hit_id + i,
                    'event_id': noise_events[i],
                    'particle_id': -1,  # Flag for noise
                    'detector': 'tracker' if np.random.random() < 0.8 else 'calorimeter',
                    'layer': layer,
                    'x': x_noise,
                    'y': y_noise,
                    'z': z_noise,
                    't': t_noise,
                    'energy': e_noise,
                    'is_noise': True
                })

        return pd.DataFrame(hits)

    def generate_decay_chains(self, particles_df, decay_probability=0.1):
        """Generate particle decay chains for unstable particles"""
        # Only unstable particles can decay
        unstable_mask = particles_df['type'].isin(['pion', 'kaon', 'muon', 'tau', 'neutron', 'Higgs', 'Z'])
        unstable_particles = particles_df[unstable_mask].copy()
        stable_particles = particles_df[~unstable_mask].copy()

        # Container for all decay products
        all_decay_products = []
        next_particle_id = particles_df['particle_id'].max() + 1

        # Only process decays for some fraction of particles
        decay_mask = np.random.random(len(unstable_particles)) < decay_probability
        decaying_particles = unstable_particles[decay_mask]
        non_decaying = unstable_particles[~decay_mask]

        # Process each decaying particle
        for _, particle in tqdm(decaying_particles.iterrows(), total=len(decaying_particles), desc="Generating decays"):
            particle_type = particle['type']
            decay_products = []

            # Different decay modes based on particle type
            if particle_type == 'pion':
                # π± → μ± + νμ  (probability ~100%)
                # Create muon
                muon_energy = particle['energy'] * np.random.uniform(0.7, 0.9)

                # Random direction in parent rest frame
                cos_theta = np.random.uniform(-1, 1)
                sin_theta = np.sqrt(1 - cos_theta**2)
                phi = np.random.uniform(0, 2*np.pi)

                # Simplified momentum calculation in parent frame
                muon_momentum = np.sqrt(muon_energy**2 - self.particle_types['muon']['mass']**2)
                muon_px = muon_momentum * sin_theta * np.cos(phi)
                muon_py = muon_momentum * sin_theta * np.sin(phi)
                muon_pz = muon_momentum * cos_theta

                # Create decay product record
                decay_products.append({
                    'particle_id': next_particle_id,
                    'parent_id': particle['particle_id'],
                    'event_id': particle['event_id'],
                    'type': 'muon',
                    'mass': self.particle_types['muon']['mass'],
                    'charge': particle['charge'],  # Same charge as parent pion
                    'px': muon_px,
                    'py': muon_py,
                    'pz': muon_pz,
                    'pt': np.sqrt(muon_px**2 + muon_py**2),
                    'energy': muon_energy,
                    'creation_x': particle['creation_x'],
                    'creation_y': particle['creation_y'],
                    'creation_z': particle['creation_z'],
                    'creation_t': particle['creation_t'],
                    'is_decay_product': True
                })
                next_particle_id += 1

            elif particle_type == 'kaon':
                # K± → π± + π0  (probability ~21%)
                # Create charged pion
                pion_energy = particle['energy'] * np.random.uniform(0.4, 0.6)

                # Random direction similar to above
                cos_theta = np.random.uniform(-1, 1)
                sin_theta = np.sqrt(1 - cos_theta**2)
                phi = np.random.uniform(0, 2*np.pi)

                pion_momentum = np.sqrt(pion_energy**2 - self.particle_types['pion']['mass']**2)
                pion_px = pion_momentum * sin_theta * np.cos(phi)
                pion_py = pion_momentum * sin_theta * np.sin(phi)
                pion_pz = pion_momentum * cos_theta

                decay_products.append({
                    'particle_id': next_particle_id,
                    'parent_id': particle['particle_id'],
                    'event_id': particle['event_id'],
                    'type': 'pion',
                    'mass': self.particle_types['pion']['mass'],
                    'charge': particle['charge'],
                    'px': pion_px,
                    'py': pion_py,
                    'pz': pion_pz,
                    'pt': np.sqrt(pion_px**2 + pion_py**2),
                    'energy': pion_energy,
                    'creation_x': particle['creation_x'],
                    'creation_y': particle['creation_y'],
                    'creation_z': particle['creation_z'],
                    'creation_t': particle['creation_t'],
                    'is_decay_product': True
                })
                next_particle_id += 1

            # Add other decay modes for different particles...
            elif particle_type == 'Higgs':
                # H → bb̄ (~58%)
                # H → WW* (~21%)
                # H → ττ (~6%)
                # H → ZZ* (~2.6%)
                # Simplified implementation: just create two b-quarks (although in reality they would hadronize)
                for i in range(2):
                    b_energy = particle['energy'] * 0.5 * np.random.uniform(0.9, 1.1)

                    # Random direction
                    cos_theta = np.random.uniform(-1, 1)
                    sin_theta = np.sqrt(1 - cos_theta**2)
                    phi = np.random.uniform(0, 2*np.pi)

                    b_momentum = np.sqrt(b_energy**2 - 4.2**2)  # b quark mass ~4.2 GeV
                    b_px = b_momentum * sin_theta * np.cos(phi)
                    b_py = b_momentum * sin_theta * np.sin(phi)
                    b_pz = b_momentum * cos_theta

                    decay_products.append({
                        'particle_id': next_particle_id,
                        'parent_id': particle['particle_id'],
                        'event_id': particle['event_id'],
                        'type': 'b_quark',  # Simplified - in reality would hadronize
                        'mass': 4.2,
                        'charge': -1/3 if i == 0 else 1/3,  # b and anti-b
                        'px': b_px,
                        'py': b_py,
                        'pz': b_pz,
                        'pt': np.sqrt(b_px**2 + b_py**2),
                        'energy': b_energy,
                        'creation_x': particle['creation_x'],
                        'creation_y': particle['creation_y'],
                        'creation_z': particle['creation_z'],
                        'creation_t': particle['creation_t'],
                        'is_decay_product': True
                    })
                    next_particle_id += 1

            # Add more decay channels as needed...

            # Append all decay products
            if decay_products:
                all_decay_products.extend(decay_products)

        # Combine original stable particles, non-decaying unstable particles, and decay products
        combined_particles = pd.concat([
            stable_particles,
            non_decaying,
            pd.DataFrame(all_decay_products) if all_decay_products else pd.DataFrame()
        ]).reset_index(drop=True)

        return combined_particles

    def generate_jet_clusters(self, particles_df, r_param=0.4, pt_min=5.0):
        """
        Simple jet clustering algorithm (very simplified anti-kT)

        Parameters:
        - r_param: Jet radius parameter
        - pt_min: Minimum pT for a jet
        """
        # Group by event
        event_ids = particles_df['event_id'].unique()
        all_jets = []
        jet_id = 0

        for event_id in tqdm(event_ids, desc="Clustering jets"):
            event_particles = particles_df[particles_df['event_id'] == event_id]

            # Skip events with too few particles
            if len(event_particles) < 3:
                continue

            # Simplified clustering (real anti-kT would be much more complex):
            # 1. Start with highest pT particle as seed
            # 2. Cluster nearby particles within r_param
            # 3. Remove clustered particles from list
            # 4. Repeat until no particles above threshold

            remaining_particles = event_particles.copy()

            while len(remaining_particles) > 0:
                # Find highest pT particle as seed
                if remaining_particles['pt'].max() < 1.0:  # Skip low pT particles
                    break

                seed_idx = remaining_particles['pt'].idxmax()
                seed = remaining_particles.loc[seed_idx]

                # Find particles within ΔR < r_param of seed
                jet_particles = []
                jet_particle_ids = []

                for idx, particle in remaining_particles.iterrows():
                    # Calculate ΔR = √(Δη² + Δφ²)
                    delta_eta = particle['eta'] - seed['eta']

                    # Handle φ periodicity
                    delta_phi = particle['phi'] - seed['phi']
                    if delta_phi > np.pi:
                        delta_phi -= 2*np.pi
                    elif delta_phi < -np.pi:
                        delta_phi += 2*np.pi

                    delta_r = np.sqrt(delta_eta**2 + delta_phi**2)

                    if delta_r < r_param:
                        jet_particles.append(particle)
                        jet_particle_ids.append(particle['particle_id'])

                # Calculate jet properties from constituent particles
                if jet_particles:
                    jet_df = pd.DataFrame(jet_particles)

                    # Basic jet four-momentum (sum of constituents)
                    jet_px = jet_df['px'].sum()
                    jet_py = jet_df['py'].sum()
                    jet_pz = jet_df['pz'].sum()
                    jet_energy = jet_df['energy'].sum()

                    jet_pt = np.sqrt(jet_px**2 + jet_py**2)

                    # Only keep jets with sufficient pT
                    if jet_pt > pt_min:
                        # Calculate other jet properties
                        jet_phi = np.arctan2(jet_py, jet_px)
                        jet_eta = np.asinh(jet_pz / jet_pt) if jet_pt > 0 else 0

                        # Calculate jet mass
                        jet_mass2 = jet_energy**2 - (jet_px**2 + jet_py**2 + jet_pz**2)
                        jet_mass = np.sqrt(max(0, jet_mass2))  # Protect against negative mass²

                        all_jets.append({
                            'jet_id': jet_id,
                            'event_id': event_id,
                            'pt': jet_pt,
                            'eta': jet_eta,
                            'phi': jet_phi,
                            'energy': jet_energy,
                            'mass': jet_mass,
                            'n_constituents': len(jet_particles),
                            'constituent_ids': jet_particle_ids
                        })
                        jet_id += 1

                # Remove clustered particles
                remaining_particles = remaining_particles[~remaining_particles['particle_id'].isin(jet_particle_ids)]

        return pd.DataFrame(all_jets)

    def generate_dataset(self, n_events=10000, output_dir="particle_data",
                         beam_energy=6500, pt_mean=1.5, efficiency=0.95,
                         decay_prob=0.2, generate_jets=True):
        """
        Generate a complete particle physics dataset

        Parameters:
        -----------
        n_events : int
            Number of collision events to generate
        output_dir : str
            Directory to save the dataset
        beam_energy : float
            Beam energy in GeV
        pt_mean : float
            Mean transverse momentum for particles
        efficiency : float
            Detector efficiency
        decay_prob : float
            Probability of unstable particle decay
        generate_jets : bool
            Whether to cluster particles into jets

        Returns:
        --------
        dict : Dictionary containing all dataframes

        start_time = time.time()

        print(f"Generating {n_events} collision events...")
        """
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)

        # Generate vertices
        vertices_df = self.generate_primary_vertex(n_events, beam_energy)

        # Generate particles
        particles_df = self.generate_particles(vertices_df, pt_mean=pt_mean)

        # Generate decays
        if decay_prob > 0:
            particles_df = self.generate_decay_chains(particles_df, decay_probability=decay_prob)

        # Generate detector hits
        hits_df = self.simulate_detector_hits(particles_df, efficiency=efficiency)

        # Generate jets
        jets_df = None
        if generate_jets:
            jets_df = self.generate_jet_clusters(particles_df)

        # Save data
        vertices_df.to_csv(f"{output_dir}/vertices.csv", index=False)
        particles_df.to_csv(f"{output_dir}/particles.csv", index=False)
        hits_df.to_csv(f"{output_dir}/hits.csv", index=False)

        if jets_df is not None:
            jets_df.to_csv(f"{output_dir}/jets.csv", index=False)

        # Also save as pickle for easier loading with pandas
        dataset = {
            'vertices': vertices_df,
            'particles': particles_df,
            'hits': hits_df
        }

        if jets_df is not None:
            dataset['jets'] = jets_df

        with open(f"{output_dir}/dataset.pkl", 'wb') as f:
            pickle.dump(dataset, f)

        end_time = time.time()
        print(f"Dataset generated in {end_time - start_time:.2f} seconds")
        print(f"Generated {len(vertices_df)} events with {len(particles_df)} particles and {len(hits_df)} detector hits")

        if jets_df is not None:
            print(f"Clustered {len(jets_df)} jets")

        return dataset

    def visualize_event(self, event_id, vertices_df, particles_df, hits_df=None, jets_df=None):
        """Visualize a single event with particle trajectories and detector hits"""
        # Filter data for this event
        event_particles = particles_df[particles_df['event_id'] == event_id]
        event_vertex = vertices_df

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import expon, norm, crystalball, gamma, pareto
from scipy.interpolate import interp1d
import os
from tqdm.notebook import tqdm
import pickle
import time
import warnings
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Union, Callable, Any
from matplotlib.patches import Circle, Rectangle
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import multiprocessing as mp
from functools import partial
import json
import h5py

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('ParticleDataGenerator')


@dataclass
class ParticleProperties:
    """Data class to store particle physics properties"""
    mass: float
    charge: float
    lifetime: float
    width: float = 0.0  # Natural width in GeV
    color_charge: bool = False
    spin: float = 0.0
    pdg_id: int = 0
    decay_modes: Dict[str, float] = field(default_factory=dict)

    def __post_init__(self):
        # Validate properties
        assert self.mass >= 0, f"Mass must be non-negative: {self.mass}"
        assert isinstance(self.charge, (int, float)), f"Charge must be numeric: {self.charge}"
        assert self.lifetime >= 0, f"Lifetime must be non-negative: {self.lifetime}"


@dataclass
class DetectorProperties:
    """Data class to store detector properties"""
    # Geometry
    barrel_radius: float = 1.1  # meters
    barrel_length: float = 5.0  # meters
    endcap_distance: float = 2.5  # meters from center
    layers: int = 5  # number of detector layers

    # Material budget in radiation lengths per layer
    material_thickness: List[float] = field(default_factory=lambda: [0.01, 0.03, 0.05, 0.10, 0.15])

    # Magnetic field
    b_field: float = 2.0  # Tesla
    b_field_direction: str = 'z'  # Direction of magnetic field

    # Resolution properties
    position_resolution: Dict[str, float] = field(
        default_factory=lambda: {'barrel': 0.01, 'endcap': 0.02}  # meters
    )
    energy_resolution: Dict[str, Dict[str, float]] = field(
        default_factory=lambda: {
            'em_calorimeter': {'stochastic': 0.10, 'constant': 0.01, 'noise': 0.1},  # EM calorimeter
            'had_calorimeter': {'stochastic': 0.50, 'constant': 0.03, 'noise': 0.5}   # Hadronic calorimeter
        }
    )
    time_resolution: float = 0.15e-9  # seconds (150 ps)

    # Tracker properties
    tracker_efficiency: Dict[str, float] = field(
        default_factory=lambda: {'electron': 0.99, 'muon': 0.98, 'pion': 0.95, 'kaon': 0.94, 'proton': 0.93}
    )

    # Calorimeter properties
    calorimeter_acceptance: Dict[str, float] = field(
        default_factory=lambda: {'eta_min': -3.0, 'eta_max': 3.0}
    )

    # Trigger thresholds
    trigger_thresholds: Dict[str, float] = field(
        default_factory=lambda: {'electron_pt': 20.0, 'muon_pt': 15.0, 'jet_pt': 30.0, 'met': 50.0}
    )

    def __post_init__(self):
        """Validate detector properties"""
        assert self.barrel_radius > 0, f"Barrel radius must be positive: {self.barrel_radius}"
        assert self.barrel_length > 0, f"Barrel length must be positive: {self.barrel_length}"
        assert len(self.material_thickness) == self.layers, (
            f"Material thickness list length ({len(self.material_thickness)}) "
            f"must match number of layers ({self.layers})"
        )


class PhysicsModels:
    """
    Class containing physics models for particle generation and interactions
    """
    @staticmethod
    def bjorken_x_distribution(x: np.ndarray, alpha: float = 0.5, beta: float = 3.0) -> np.ndarray:
        """Compute parton distribution function (simplified)"""
        return np.power(x, -alpha) * np.power(1-x, beta)

    @staticmethod
    def pt_spectrum(pt: np.ndarray, p0: float = 2.0, n: float = 5.0) -> np.ndarray:
        """Compute pT spectrum using power law distribution"""
        return np.power(1.0 + pt/p0, -n)

    @staticmethod
    def crystal_ball_mass(m: np.ndarray, m0: float, sigma: float,
                         alpha: float = 1.0, n: float = 3.0) -> np.ndarray:
        """Crystal Ball function for resonance mass distribution"""
        return crystalball.pdf(m, alpha, n, loc=m0, scale=sigma)

    @staticmethod
    def breit_wigner(m: np.ndarray, m0: float, gamma: float) -> np.ndarray:
        """Relativistic Breit-Wigner distribution for resonance mass"""
        return gamma / (2 * np.pi * ((m - m0)**2 + (gamma/2)**2))

    @staticmethod
    def fragmentation_function(z: np.ndarray, a: float = 0.3, b: float = 0.58) -> np.ndarray:
        """Fragmentation function for quark hadronization"""
        return (1/z) * np.power(1-z, a) * np.exp(-b * np.square(1/z))

    @staticmethod
    def decay_probability(lifetime: float, proper_time: float) -> float:
        """Calculate decay probability based on particle lifetime"""
        if lifetime == np.inf:
            return 0.0

        # Exponential decay law
        return 1.0 - np.exp(-proper_time / lifetime)


class ParticleDataGenerator:
    """
    Enhanced generator for synthetic particle physics data with configurable properties.
    """

    def __init__(self, detector_config: Optional[Dict] = None,
                 particle_config: Optional[Dict] = None, seed: int = 42):
        """Initialize the particle data generator"""
        # Set random seed
        np.random.seed(seed)
        self.seed = seed

        # Initialize physics models
        self.physics = PhysicsModels()

        # Initialize particle properties
        self._init_particle_properties(particle_config)

        # Initialize detector properties
        self._init_detector_properties(detector_config)

        # Physics constants
        self.constants = {
            'c': 299792458,   # Speed of light in m/s
            'hbar': 6.582e-25, # GeV·s
            'alpha_em': 1/137, # Fine structure constant
            'G_F': 1.166e-5,   # Fermi constant in GeV^-2
        }

        # Track counters for unique IDs
        self.event_counter = 0
        self.vertex_counter = 0
        self.particle_counter = 0
        self.hit_counter = 0
        self.jet_counter = 0

        # Logging setup
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"ParticleDataGenerator initialized with seed {seed}")

In [ ]:
    def run_simulation(self, n_events: int = 1000, particles_per_vertex: int = 10,
                       output_dir: str = 'output', output_format: str = 'csv') -> None:
        """
        Run the full simulation for a specified number of events

        Parameters:
        -----------
        n_events : int
            Number of events to simulate
        particles_per_vertex : int
            Number of particles to generate per primary vertex per event
        output_dir : str
            Directory to save the output files
        output_format : str
            Format to save the output ('csv')
        """
        # Validate parameters
        self._validate_parameters(
            {'n_events': n_events, 'particles_per_vertex': particles_per_vertex,
             'output_dir': output_dir, 'output_format': output_format},
            ['n_events', 'particles_per_vertex', 'output_dir', 'output_format'],
            {'n_events': int, 'particles_per_vertex': int, 'output_dir': str, 'output_format': str},
            {'n_events': (1, 1e6), 'particles_per_vertex': (1, 100)}
        )
        if output_format != 'csv':
            raise ValueError(f"Unsupported output format: {output_format}. Please choose 'csv'.")

        os.makedirs(output_dir, exist_ok=True)
        start_time = time.time()
        self.logger.info(f"Starting simulation of {n_events} events...")

        all_vertices = []
        all_particles = []
        all_hits = []
        all_tracks = []
        all_clusters = []
        all_jets = []

        for i in tqdm(range(n_events), desc="Simulating Events"):
            vertices, particles, hits, tracks, clusters, jets = self.simulate_event(n_particles=particles_per_vertex)
            all_vertices.append(vertices)
            all_particles.append(particles)
            all_hits.append(hits)
            all_tracks.append(tracks)
            all_clusters.append(clusters)
            all_jets.append(jets)

        # Concatenate results
        if all_vertices:
            vertices_df = pd.concat(all_vertices, ignore_index=True)
            particles_df = pd.concat(all_particles, ignore_index=True)
            hits_df = pd.concat(all_hits, ignore_index=True)
            tracks_df = pd.concat(all_tracks, ignore_index=True)
            clusters_df = pd.concat(all_clusters, ignore_index=True)
            jets_df = pd.concat(all_jets, ignore_index=True)

            # Save to CSV format
            base_filename = os.path.join(output_dir, f"simulation_data")
            vertices_df.to_csv(f"{base_filename}_vertices.csv", index=False)
            particles_df.to_csv(f"{base_filename}_particles.csv", index=False)
            hits_df.to_csv(f"{base_filename}_hits.csv", index=False)
            tracks_df.to_csv(f"{base_filename}_tracks.csv", index=False)
            clusters_df.to_csv(f"{base_filename}_clusters.csv", index=False)
            jets_df.to_csv(f"{base_filename}_jets.csv", index=False)
            self.logger.info(f"Simulation data saved to {output_dir} in CSV format.")

        else:
            self.logger.warning("No events were generated.")

        end_time = time.time()
        duration = end_time - start_time
        self.logger.info(f"Simulation finished in {duration:.2f} seconds.")